In [ ]:

import numpy as np
import pandas as pd

from datetime import datetime, timedelta


import duckdb


DB_PATH = r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\duckdb\nhl_data.duckdb"

con = duckdb.connect(DB_PATH)
print("Connected:", DB_PATH)


# Suppress SettingWithCopyWarning
pd.options.mode.chained_assignment = None
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

Connected: C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\duckdb\nhl_data.duckdb


In [ ]:

# con.close()
# print("DuckDB connection closed.")

DuckDB connection closed.


In [112]:

con.execute("SHOW TABLES").fetchdf()

,name
0,all_clinch_history
1,all_final_standings
2,all_game_summary
3,all_goalie_games
4,all_pk
5,all_play_by_play
6,all_play_by_play_1
7,all_pp
8,all_prediction_data_agg
9,all_standings_daily


In [4]:
teams = con.execute('select * from all_teams').df()
teams.head()

,team_id,tri_code
0,24,ANA
1,53,ARI
2,6,BOS
3,7,BUF
4,12,CAR


## Game Summary

In [5]:
history_game_summary=con.execute("""SELECT *  
                                 FROM all_game_summary 
                                 WHERE season_id != 20252026
                                 ORDER BY game_id""").df()

history_game_summary.tail()

,game_id,season_id,away_team,away_tri_code,home_team,home_tri_code,awayTeam.score,homeTeam.score,gameState,link,game_date
14646,2024021308,20242025,17,DET,10,TOR,3,4,OFF,https://api-web.nhle.com/v1/gamecenter/2024021308/play-by-play,2025-04-17
14647,2024021309,20242025,12,CAR,9,OTT,5,7,OFF,https://api-web.nhle.com/v1/gamecenter/2024021309/play-by-play,2025-04-17
14648,2024021310,20242025,14,TBL,3,NYR,0,4,OFF,https://api-web.nhle.com/v1/gamecenter/2024021310/play-by-play,2025-04-17
14649,2024021311,20242025,15,WSH,5,PIT,2,5,OFF,https://api-web.nhle.com/v1/gamecenter/2024021311/play-by-play,2025-04-17
14650,2024021312,20242025,2,NYI,29,CBJ,1,6,OFF,https://api-web.nhle.com/v1/gamecenter/2024021312/play-by-play,2025-04-17


In [ ]:
game_summary=con.execute("""SELECT *  
                         FROM all_game_summary
                         WHERE season_id = 20252026 
                         ORDER BY game_id""").df()

game_summary.tail()

,game_id,season_id,away_team,away_tri_code,home_team,home_tri_code,awayTeam.score,homeTeam.score,gameState,link,game_date
1307,2025021308,20252026,28,SJS,52,WPG,<NA>,<NA>,FUT,https://api-web.nhle.com/v1/gamecenter/2025021308/play-by-play,2026-04-16
1308,2025021309,20252026,19,STL,68,UTA,<NA>,<NA>,FUT,https://api-web.nhle.com/v1/gamecenter/2025021309/play-by-play,2026-04-16
1309,2025021310,20252026,26,LAK,20,CGY,<NA>,<NA>,FUT,https://api-web.nhle.com/v1/gamecenter/2025021310/play-by-play,2026-04-16
1310,2025021311,20252026,23,VAN,22,EDM,<NA>,<NA>,FUT,https://api-web.nhle.com/v1/gamecenter/2025021311/play-by-play,2026-04-16
1311,2025021312,20252026,55,SEA,21,COL,<NA>,<NA>,FUT,https://api-web.nhle.com/v1/gamecenter/2025021312/play-by-play,2026-04-16


In [116]:
clinch_date = con.execute("""
                          SELECT
                          season_id,
                          clinch_date as game_date,
                          tri_code,
                          clinch_type
                          FROM all_clinch_history 
                          ORDER BY season_id""").df()
clinch_outcomes = clinch_date.assign(
    made_playoffs=lambda df: df["clinch_type"].eq("x").astype(int)
)




## Standings

In [117]:
history_daily_standings=con.execute("""
                                    SELECT *  
                                    FROM all_standings_daily
                                    WHERE season_id !=20252026""").df()


history_daily_standings=history_daily_standings.merge(clinch_date, on=['season_id', 'game_date', 'tri_code'], how='left')

history_daily_standings=history_daily_standings.merge(
    clinch_outcomes[["season_id", "tri_code", "made_playoffs"]],
    on=["season_id", "tri_code"],
    how="left"
)
history_daily_standings=history_daily_standings[[
    'game_date', 'season_id', 'team_id','tri_code', 'team_name',
    
    'games_remaining', 
    'wins', 'wins_pctg', 'losses', 'ot_losses',
    'rotw', 'rotw_pctg', 'rw',  'so_losses', 'so_wins',
    
    'points', 'points_pct','projected_points', 'in_playoff_spot', 
    'goals_for', 'gf_pctg','goals_against',
    'goal_diff', 'goal_diff_pctg',
    'streak', 'streak_count', 
    
    'last_10_gp','last_10_wins', 'last_10_losses', 'last_10_otl', 
    'last_10_points', 
    'last_10_gf', 'last_10_ga', 'last_10_goal_diff', 'last_10_rotw',
    'last_10_rw', 
     
       
    'home_gp', 
    'home_wins', 'home_losses', 'home_otlosses',
    'home_rotw', 'home_rw',
    'home_points', 'home_gf','home_ga',  'home_goal_diff',
    
    'road_gp', 
    'road_wins', 'road_losses', 'road_ot_losses', 
    'road_rotw','road_rw',
    'road_points','road_gf', 'road_ga',  'road_goal_diff', 
    
    'division', 'division_rank',
    'division_home_rank', 'division_road_rank', 'division_last_10_rank',
    
    'conference', 'conference_rank', 'conference_home_rank',
    'conference_road_rank', 'conference_last_10_rank', 
    
    'league_rank',
    'league_home_rank', 'league_road_rank', 'league_last_10_rank',
    
    'wildcard_rank', "made_playoffs"
    
]]
history_daily_standings['made_playoffs'] = history_daily_standings['made_playoffs'].astype("Int64")
history_daily_standings.head(2)



,game_date,season_id,team_id,tri_code,team_name,games_remaining,wins,wins_pctg,losses,ot_losses,rotw,rotw_pctg,rw,so_losses,so_wins,points,points_pct,projected_points,in_playoff_spot,goals_for,gf_pctg,goals_against,goal_diff,goal_diff_pctg,streak,streak_count,last_10_gp,last_10_wins,last_10_losses,last_10_otl,last_10_points,last_10_gf,last_10_ga,last_10_goal_diff,last_10_rotw,last_10_rw,home_gp,home_wins,home_losses,home_otlosses,home_rotw,home_rw,home_points,home_gf,home_ga,home_goal_diff,road_gp,road_wins,road_losses,road_ot_losses,road_rotw,road_rw,road_points,road_gf,road_ga,road_goal_diff,division,division_rank,division_home_rank,division_road_rank,division_last_10_rank,conference,conference_rank,conference_home_rank,conference_road_rank,conference_last_10_rank,league_rank,league_home_rank,league_road_rank,league_last_10_rank,wildcard_rank,made_playoffs
0,2013-10-01,20132014,16,CHI,Chicago Blackhawks,81,1,1.0,0,0,1,1.0,1,0,0,2,1.0,164,0,6,6.0,4,2,2.0,W,1.0,1,1,0,0,2,6,4,2,1,1,1,1,0,0,1,1,2,6,4,2,0,0,0,0,0,0,0,0,0,0,Central,1,1,2,1,Western,1,1,4,1,1,1,8,1,0,1
1,2013-10-01,20132014,10,TOR,Toronto Maple Leafs,81,1,1.0,0,0,1,1.0,1,0,0,2,1.0,164,0,4,4.0,3,1,1.0,W,1.0,1,1,0,0,2,4,3,1,1,1,0,0,0,0,0,0,0,0,0,0,1,1,0,0,1,1,2,4,3,1,Atlantic,1,7,1,1,Eastern,1,14,1,1,2,25,1,2,0,0


In [118]:
daily_standings=con.execute("""SELECT *  
                            FROM all_standings_daily
                            WHERE season_id =20252026""").df()

daily_standings.head()


,game_date,date,season_id,tri_code,team_name,games_played,road_gp,division,division_rank,division_home_rank,division_road_rank,division_last_10_rank,conference,conference_rank,conference_home_rank,conference_road_rank,conference_last_10_rank,league_rank,league_home_rank,league_road_rank,league_last_10_rank,wildcard_rank,wins,wins_pctg,rw,losses,ot_losses,rotw,rotw_pctg,road_losses,road_ot_losses,road_rotw,road_rw,points,points_pct,home_points,road_points,goals_for,gf_pctg,home_gf,road_gf,goals_against,home_ga,road_ga,goal_diff,goal_diff_pctg,home_goal_diff,road_goal_diff,streak,streak_count,last_10_gp,last_10_wins,last_10_goal_diff,last_10_ga,last_10_gf,last_10_losses,last_10_otl,last_10_points,last_10_rotw,last_10_rw,home_gp,home_wins,home_losses,home_otlosses,home_rotw,home_rw,road_wins,so_losses,so_wins,in_playoff_spot,games_remaining,ppg,projected_points,team_id
0,2025-10-07,2025-10-07,20252026,COL,Colorado Avalanche,1,1,Central,1,2,1,1,Western,1,4,1,1,1,8,1,1,0,1,1.0,1,0,0,1,1.0,0,0,1,1,2,1.0,0,2,4,4.0,0,4,1,0,1,3,3.0,0,3,W,1.0,1,1,3,1,4,0,0,2,1,1,0,0,0,0,0,0,1,0,0,0,81,2.0,164,21
1,2025-10-07,2025-10-07,20252026,PIT,Pittsburgh Penguins,1,1,Metropolitan,1,6,1,1,Eastern,1,12,1,1,2,20,2,2,0,1,1.0,1,0,0,1,1.0,0,0,1,1,2,1.0,0,2,3,3.0,0,3,0,0,0,3,3.0,0,3,W,1.0,1,1,3,0,3,0,0,2,1,1,0,0,0,0,0,0,1,0,0,0,81,2.0,164,5
2,2025-10-07,2025-10-07,20252026,FLA,Florida Panthers,1,0,Atlantic,1,1,4,1,Eastern,2,1,7,2,3,1,12,3,0,1,1.0,1,0,0,1,1.0,0,0,0,0,2,1.0,2,0,3,3.0,3,0,2,2,0,1,1.0,1,0,W,1.0,1,1,1,2,3,0,0,2,1,1,1,1,0,0,1,1,0,0,0,0,81,2.0,164,13
3,2025-10-07,2025-10-07,20252026,ANA,Anaheim Ducks,0,0,Pacific,1,1,1,1,Western,2,1,2,2,4,2,3,4,0,0,0.0,0,0,0,0,0.0,0,0,0,0,0,0.0,0,0,0,0.0,0,0,0,0,0,0,0.0,0,0,None,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,82,NaN,0,24
4,2025-10-07,2025-10-07,20252026,BOS,Boston Bruins,0,0,Atlantic,2,2,1,2,Eastern,3,2,2,3,5,3,4,5,0,0,0.0,0,0,0,0,0.0,0,0,0,0,0,0.0,0,0,0,0.0,0,0,0,0,0,0,0.0,0,0,None,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,82,NaN,0,6


In [119]:
daily_standings=daily_standings[[
    'game_date', 'season_id', 'team_id','tri_code', 'team_name',
    
    'games_remaining', 
    'wins', 'wins_pctg', 'losses', 'ot_losses',
    'rotw', 'rotw_pctg', 'rw',  'so_losses', 'so_wins',
    
    'points', 'points_pct','projected_points', 'in_playoff_spot', 
    'goals_for', 'gf_pctg','goals_against',
    'goal_diff', 'goal_diff_pctg',
    'streak', 'streak_count', 
    
    'last_10_gp','last_10_wins', 'last_10_losses', 'last_10_otl', 
    'last_10_points', 
    'last_10_gf', 'last_10_ga', 'last_10_goal_diff', 'last_10_rotw',
    'last_10_rw', 
     
       
    'home_gp', 
    'home_wins', 'home_losses', 'home_otlosses',
    'home_rotw', 'home_rw',
    'home_points', 'home_gf','home_ga',  'home_goal_diff',
    
    'road_gp', 
    'road_wins', 'road_losses', 'road_ot_losses', 
    'road_rotw','road_rw',
    'road_points','road_gf', 'road_ga',  'road_goal_diff', 
    
    'division', 'division_rank',
    'division_home_rank', 'division_road_rank', 'division_last_10_rank',
    
    'conference', 'conference_rank', 'conference_home_rank',
    'conference_road_rank', 'conference_last_10_rank', 
    
    'league_rank',
    'league_home_rank', 'league_road_rank', 'league_last_10_rank',
    
    'wildcard_rank',     
    
]]

daily_standings.head(2)

,game_date,season_id,team_id,tri_code,team_name,games_remaining,wins,wins_pctg,losses,ot_losses,rotw,rotw_pctg,rw,so_losses,so_wins,points,points_pct,projected_points,in_playoff_spot,goals_for,gf_pctg,goals_against,goal_diff,goal_diff_pctg,streak,streak_count,last_10_gp,last_10_wins,last_10_losses,last_10_otl,last_10_points,last_10_gf,last_10_ga,last_10_goal_diff,last_10_rotw,last_10_rw,home_gp,home_wins,home_losses,home_otlosses,home_rotw,home_rw,home_points,home_gf,home_ga,home_goal_diff,road_gp,road_wins,road_losses,road_ot_losses,road_rotw,road_rw,road_points,road_gf,road_ga,road_goal_diff,division,division_rank,division_home_rank,division_road_rank,division_last_10_rank,conference,conference_rank,conference_home_rank,conference_road_rank,conference_last_10_rank,league_rank,league_home_rank,league_road_rank,league_last_10_rank,wildcard_rank
0,2025-10-07,20252026,21,COL,Colorado Avalanche,81,1,1.0,0,0,1,1.0,1,0,0,2,1.0,164,0,4,4.0,1,3,3.0,W,1.0,1,1,0,0,2,4,1,3,1,1,0,0,0,0,0,0,0,0,0,0,1,1,0,0,1,1,2,4,1,3,Central,1,2,1,1,Western,1,4,1,1,1,8,1,1,0
1,2025-10-07,20252026,5,PIT,Pittsburgh Penguins,81,1,1.0,0,0,1,1.0,1,0,0,2,1.0,164,0,3,3.0,0,3,3.0,W,1.0,1,1,0,0,2,3,0,3,1,1,0,0,0,0,0,0,0,0,0,0,1,1,0,0,1,1,2,3,0,3,Metropolitan,1,6,1,1,Eastern,1,12,1,1,2,20,2,2,0


## Play by Play

In [120]:
history_pbp=con.execute(""" SELECT
                    game_id,
                    team_id,
                    home_team,
                    away_team,
                    type_code,
                    typeDescKey,
                    zone_code,
                    x_coord,
                    y_coord,
                    period_number,
                    game_in_seconds,
                    goalie,
                    sortOrder,
                    season_id
                    from all_play_by_play 
                    where season_id !=20252026
                    order by game_id, sortOrder
                    """).df()
history_pbp.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,game_id,team_id,home_team,away_team,type_code,typeDescKey,zone_code,x_coord,y_coord,period_number,game_in_seconds,goalie,sortOrder,season_id
0,2013020001,<NA>,8,10,520,period-start,None,NaN,NaN,1,0,<NA>,5,20132014
1,2013020001,10,8,10,502,faceoff,N,0.0,0.0,1,0,<NA>,12,20132014
2,2013020001,10,8,10,507,missed-shot,O,35.0,-28.0,1,74,8471679,13,20132014
3,2013020001,8,8,10,506,shot-on-goal,O,-82.0,-22.0,1,96,8473503,18,20132014
4,2013020001,8,8,10,503,hit,O,-97.0,20.0,1,100,<NA>,19,20132014


In [121]:
current_pbp=con.execute(""" SELECT
                    game_id,
                    team_id,
                    home_team,
                    away_team,
                    type_code,
                    typeDescKey,
                    zone_code,
                    x_coord,
                    y_coord,
                    period_number,
                    game_in_seconds,
                    goalie,
                    sortOrder,
                    season_id
                    from all_play_by_play 
                    where season_id ==20252026
                    order by game_id, sortOrder
                    """).df()
current_pbp.head()

,game_id,team_id,home_team,away_team,type_code,typeDescKey,zone_code,x_coord,y_coord,period_number,game_in_seconds,goalie,sortOrder,season_id
0,2025020001,<NA>,13,16,520,period-start,None,NaN,NaN,1,0,<NA>,8,20252026
1,2025020001,16,13,16,502,faceoff,N,0.0,0.0,1,0,<NA>,11,20252026
2,2025020001,13,13,16,508,blocked-shot,D,-61.0,3.0,1,22,<NA>,12,20252026
3,2025020001,<NA>,13,16,516,stoppage,None,NaN,NaN,1,34,<NA>,13,20252026
4,2025020001,16,13,16,502,faceoff,D,-69.0,22.0,1,34,<NA>,16,20252026


In [122]:
current_pbp=con.execute(""" SELECT
                    game_id,
                    team_id,
                    home_team,
                    away_team,
                    type_code,
                    typeDescKey,
                    zone_code,
                    x_coord,
                    y_coord,
                    period_number,
                    game_in_seconds,
                    goalie,
                    sortOrder,
                    season_id
                    from all_play_by_play 
                    where season_id ==20252026
                    order by game_id, sortOrder
                    """).df()
current_pbp.head()

,game_id,team_id,home_team,away_team,type_code,typeDescKey,zone_code,x_coord,y_coord,period_number,game_in_seconds,goalie,sortOrder,season_id
0,2025020001,<NA>,13,16,520,period-start,None,NaN,NaN,1,0,<NA>,8,20252026
1,2025020001,16,13,16,502,faceoff,N,0.0,0.0,1,0,<NA>,11,20252026
2,2025020001,13,13,16,508,blocked-shot,D,-61.0,3.0,1,22,<NA>,12,20252026
3,2025020001,<NA>,13,16,516,stoppage,None,NaN,NaN,1,34,<NA>,13,20252026
4,2025020001,16,13,16,502,faceoff,D,-69.0,22.0,1,34,<NA>,16,20252026


### Rush

In [123]:
def build_rush_stats(df):
    """
    Unified rush-shot classifier and rush-shot stats generator.
    Works for both historical and current play-by-play datasets.
    """

    df = df.copy()

    # --------------------------------------
    # 1. Define event groups
    # --------------------------------------
    shot_events = {
        "shot-on-goal",
        "missed-shot",
        "goal",
        "blocked-shot",
        "failed-shot-attempt",
    }

    transition_events = {"giveaway", "takeaway", "hit"}

    stoppage_events = {
        "faceoff",
        "stoppage",
        "period-end",
        "game-end",
        "penalty",
        "delayed-penalty",
    }

    valid_prev_events = shot_events | transition_events | stoppage_events

    # --------------------------------------
    # 2. Filter to relevant events
    # --------------------------------------
    plays = df[df["typeDescKey"].isin(valid_prev_events)].copy()
    plays = plays.sort_values(["game_id", "sortOrder"])

    # --------------------------------------
    # 3. Build previous-event context
    # --------------------------------------
    g = plays.groupby("game_id", sort=False)

    plays["prev_event_time"] = g["game_in_seconds"].shift(1)
    plays["prev_event_type"] = g["typeDescKey"].shift(1)
    plays["prev_zone"] = g["zone_code"].shift(1)
    plays["prev_team_id"] = g["team_id"].shift(1)

    plays["prev_event_was_stoppage"] = plays["prev_event_type"].isin(stoppage_events)
    plays["time_diff"] = plays["game_in_seconds"] - plays["prev_event_time"]

    # --------------------------------------
    # 4. Rush classification
    # --------------------------------------
    is_shot = plays["typeDescKey"].isin(shot_events)
    quick = plays["time_diff"].between(0, 5, inclusive="both")

    possession_change = (
        plays["prev_team_id"].notna()
        & (plays["prev_team_id"] != plays["team_id"])
    )

    same_team_from_dz_nz = (
        (plays["prev_team_id"] == plays["team_id"])
        & plays["prev_zone"].isin(["D", "N"])
    )

    no_stoppage = ~plays["prev_event_was_stoppage"]

    rush_mask = (
        is_shot
        & quick
        & no_stoppage
        & (possession_change | same_team_from_dz_nz)
    )

    rush_events = plays.loc[rush_mask].copy().drop_duplicates()

    
    
    
    rush_events["rush_shot"] = 1

    # --------------------------------------
    # 5. Rush shots FOR
    # --------------------------------------
    rush_for = (
        rush_events
        .groupby(["season_id", "game_id", "team_id"])
        .size()
        .reset_index(name="rush_shots_for")
    )

    # --------------------------------------
    # 6. Rush shots AGAINST
    # --------------------------------------
    rush_against = (
        rush_events
        .groupby(["season_id", "game_id", "prev_team_id"])
        .size()
        .reset_index(name="rush_shots_against")
        .rename(columns={"prev_team_id": "team_id"})
    )

    # --------------------------------------
    # 7. Merge FOR + AGAINST
    # --------------------------------------
    rush_stats = (
        rush_for
        .merge(rush_against, on=["season_id", "game_id", "team_id"], how="inner")
        .fillna(0)
    )

    rush_stats["rush_shots_for"] = rush_stats["rush_shots_for"].astype(int)
    rush_stats["rush_shots_against"] = rush_stats["rush_shots_against"].astype(int)

    rush_stats["rush_shot_differential"] = (
        rush_stats["rush_shots_for"] - rush_stats["rush_shots_against"]
    )

    rush_stats = (
        rush_stats
        .sort_values(["game_id", "rush_shot_differential"], ascending=[True, False])
        .reset_index(drop=True)
    )

    return rush_events, rush_stats


In [124]:

# Usage
hist_rush_events, hist_rush_stats = build_rush_stats(history_pbp)
curr_rush_events, curr_rush_stats = build_rush_stats(current_pbp)


In [125]:
curr_rush_events.head()

,game_id,team_id,home_team,away_team,type_code,typeDescKey,zone_code,x_coord,y_coord,period_number,game_in_seconds,goalie,sortOrder,season_id,prev_event_time,prev_event_type,prev_zone,prev_team_id,prev_event_was_stoppage,time_diff,rush_shot
7,2025020001,13,13,16,508,blocked-shot,D,-64.0,-16.0,1,73,<NA>,25,20252026,70.0,giveaway,N,16,False,3.0,1
31,2025020001,13,13,16,506,shot-on-goal,O,-58.0,28.0,1,214,8481519,73,20252026,211.0,blocked-shot,D,13,False,3.0,1
40,2025020001,13,13,16,507,missed-shot,O,-28.0,18.0,1,292,8481519,91,20252026,288.0,hit,D,16,False,4.0,1
69,2025020001,13,13,16,508,blocked-shot,D,-59.0,35.0,1,629,<NA>,171,20252026,625.0,hit,O,16,False,4.0,1
71,2025020001,13,13,16,507,missed-shot,O,-33.0,39.0,1,638,8481519,173,20252026,634.0,hit,D,16,False,4.0,1


In [126]:
curr_rush_stats.head()

,season_id,game_id,team_id,rush_shots_for,rush_shots_against,rush_shot_differential
0,20252026,2025020001,13,12,6,6
1,20252026,2025020001,16,2,8,-6
2,20252026,2025020002,5,9,7,2
3,20252026,2025020002,3,7,9,-2
4,20252026,2025020003,21,7,6,1


##### Cumulative Rush Stats

In [127]:
def cumulative_rush(df):
    rush_to_date = df.copy()
    rush_to_date = rush_to_date.drop_duplicates()
    # Ensure correct ordering
    rush_to_date = rush_to_date.sort_values(
        ["season_id", "team_id", "game_date", "game_id"]
        if "game_date" in rush_to_date.columns
        else ["season_id", "team_id", "game_id"]
    )

    # Group by season + team (NOT game)
    g = rush_to_date.groupby(["season_id", "team_id"])

    # Cumulative totals
    rush_to_date["c_rush_shots_for"] = g["rush_shots_for"].cumsum()
    rush_to_date["c_rush_shots_against"] = g["rush_shots_against"].cumsum()
    rush_to_date["c_rush_diff"] = g["rush_shot_differential"].cumsum()

    # Games played to date
    rush_to_date["games_played"] = g.cumcount() + 1

    # Per-game rates
    rush_to_date["rush_for_per_game"] = (
        rush_to_date["c_rush_shots_for"] / rush_to_date["games_played"]
    )

    rush_to_date["rush_against_per_game"] = (
        rush_to_date["c_rush_shots_against"] / rush_to_date["games_played"]
    )

    rush_to_date["rush_diff_per_game"] = (
        rush_to_date["c_rush_diff"] / rush_to_date["games_played"]
    )

    return rush_to_date



In [128]:
current_cumulative_rush = cumulative_rush(curr_rush_stats)
history_cumulative_rush = cumulative_rush(hist_rush_stats)

history_cumulative_rush.head()

,season_id,game_id,team_id,rush_shots_for,rush_shots_against,rush_shot_differential,c_rush_shots_for,c_rush_shots_against,c_rush_diff,games_played,rush_for_per_game,rush_against_per_game,rush_diff_per_game
14,20132014,2013020008,1,6,4,2,6,4,2,1,6.000000,4.000000,2.000000
31,20132014,2013020016,1,7,9,-2,13,13,0,2,6.500000,6.500000,0.000000
68,20132014,2013020035,1,7,3,4,20,16,4,3,6.666667,5.333333,1.333333
122,20132014,2013020063,1,6,6,0,26,22,4,4,6.500000,5.500000,1.000000
151,20132014,2013020077,1,5,6,-1,31,28,3,5,6.200000,5.600000,0.600000


### All Shot Attempts

In [129]:
def extract_shot_attempts(pbp_df):
    SHOT_CODES = [505, 506, 507, 508, 537]

    # -------------------------------------------------
    # 1. Filter to shot attempts (PBP-level)
    # -------------------------------------------------
    shot_events = pbp_df.copy()
    shot_events = shot_events[shot_events["type_code"].isin(SHOT_CODES)].copy()
    shot_events = shot_events.sort_values(["game_id", "sortOrder"])

    shot_events["shot_attempt"] = 1

    # Ensure ordering for prev_team logic

    # -------------------------------------------------
    # 2. Previous team (for against calculation)
    # -------------------------------------------------
    g = shot_events.groupby("game_id")
    shot_events["prev_team_id"] = g["team_id"].shift(1)

    # # -------------------------------------------------
    # # 3. Merge rush flags
    # # -------------------------------------------------
    # rush_small = rush_events_df[
    #     ["game_id", "sortOrder", "game_in_seconds", "rush_shot"]
    # ].copy()

    # shot_events = shot_events.merge(
    #     rush_small,
    #     on=["game_id", "sortOrder", "game_in_seconds"],
    #     how="left"
    # )

    # shot_events["rush_shot"] = shot_events["rush_shot"].fillna(0).astype(int)

    # =================================================
    # GAME-LEVEL AGGREGATION (MINI DF)
    # =================================================

    # -------------------------------------------------
    # 4. Shot attempts FOR
    # -------------------------------------------------
    shots_for = (
        shot_events.groupby(["season_id", "game_id", "team_id"])
                 .size()
                 .reset_index(name="shot_attempts_for")
    )

    # -------------------------------------------------
    # 5. Shot attempts AGAINST
    # -------------------------------------------------
    shots_against = (
        shot_events[shot_events["prev_team_id"].notna()]
        .groupby(["season_id", "game_id", "prev_team_id"])
        .size()
        .reset_index(name="shot_attempts_against")
        .rename(columns={"prev_team_id": "team_id"})
    )

    # -------------------------------------------------
    # 6. Merge + differential
    # -------------------------------------------------
    shot_stats = (
        shots_for
        .merge(
            shots_against,
            on=["season_id", "game_id", "team_id"],
            how="outer"
        )
        .fillna(0)
    )

    shot_stats["shot_attempts_for"] = shot_stats["shot_attempts_for"].astype(int)
    shot_stats["shot_attempts_against"] = shot_stats["shot_attempts_against"].astype(int)

    shot_stats["shot_attempts_differential"] = (
        shot_stats["shot_attempts_for"] - shot_stats["shot_attempts_against"]
    )

    # -------------------------------------------------
    # 7. Corsi For / Against
    # -------------------------------------------------
    corsi_for = (
        shot_events[shot_events["zone_code"] == "O"]
        .groupby(["season_id", "game_id", "team_id"])
        .size()
        .reset_index(name="corsi_for")
    )

    corsi_against = (
        shot_events[
            (shot_events["zone_code"] == "O") &
            (shot_events["prev_team_id"].notna())
        ]
        .groupby(["season_id", "game_id", "prev_team_id"])
        .size()
        .reset_index(name="corsi_against")
        .rename(columns={"prev_team_id": "team_id"})
    )
    

    shot_stats = (
        shot_stats
        .merge(corsi_for, on=["season_id", "game_id", "team_id"], how="left")
        .merge(corsi_against, on=["season_id", "game_id", "team_id"], how="left")
        .fillna(0)
    )

    shot_stats['corsi_diff'] = shot_stats['corsi_for']-shot_stats['corsi_against']
    # -------------------------------------------------
    # RETURN BOTH
    # -------------------------------------------------
    return shot_stats, shot_events


In [130]:

hist_shots_stats, hist_shot_events = extract_shot_attempts(history_pbp)
curr_shots_stats, curr_shot_events     = extract_shot_attempts(current_pbp)
curr_shot_events.head()

,game_id,team_id,home_team,away_team,type_code,typeDescKey,zone_code,x_coord,y_coord,period_number,game_in_seconds,goalie,sortOrder,season_id,shot_attempt,prev_team_id
2,2025020001,13,13,16,508,blocked-shot,D,-61.0,3.0,1,22,<NA>,12,20252026,1,<NA>
7,2025020001,13,13,16,508,blocked-shot,D,-64.0,-16.0,1,73,<NA>,25,20252026,1,13
10,2025020001,13,13,16,506,shot-on-goal,O,-58.0,-22.0,1,76,8481519,29,20252026,1,13
15,2025020001,13,13,16,506,shot-on-goal,O,-33.0,-19.0,1,95,8481519,35,20252026,1,13
18,2025020001,16,13,16,507,missed-shot,O,55.0,-29.0,1,104,8475683,38,20252026,1,13


In [131]:
hist_shots_stats.head()

,season_id,game_id,team_id,shot_attempts_for,shot_attempts_against,shot_attempts_differential,corsi_for,corsi_against,corsi_diff
0,20132014,2013020001,8,70,69,1,53.0,53.0,0.0
1,20132014,2013020001,10,61,61,0,45.0,44.0,1.0
2,20132014,2013020002,15,57,57,0,39.0,39.0,0.0
3,20132014,2013020002,16,55,54,1,38.0,38.0,0.0
4,20132014,2013020003,22,71,70,1,48.0,49.0,-1.0


In [132]:
curr_shots_stats[curr_shots_stats['game_id']==2025020014]

,season_id,game_id,team_id,shot_attempts_for,shot_attempts_against,shot_attempts_differential,corsi_for,corsi_against,corsi_diff
26,20252026,2025020014,1,45,45,0,30,33,-3
27,20252026,2025020014,12,73,72,1,51,47,4


#### Cumulative Shot Attempts

In [133]:
def cumulative_shot_attempts(df):
    shots_to_date = df.copy()

    # -------------------------------------------------
    # Ensure required columns exist
    # -------------------------------------------------
    required_cols = [
        "shot_attempts_for", "shot_attempts_against",
        "shot_attempts_differential",
        "corsi_for", "corsi_against"
    ]
    for col in required_cols:
        if col not in shots_to_date.columns:
            shots_to_date[col] = 0

    # -------------------------------------------------
    # Ensure correct ordering
    # -------------------------------------------------
    shots_to_date = shots_to_date.sort_values(
        ["season_id", "team_id", "game_id"]
    )

    # -------------------------------------------------
    # Group by season + team
    # -------------------------------------------------
    g = shots_to_date.groupby(["season_id", "team_id"])

    # -------------------------------------------------
    # Cumulative totals
    # -------------------------------------------------
    shots_to_date["c_shots_for"] = g["shot_attempts_for"].cumsum()
    shots_to_date["c_shots_against"] = g["shot_attempts_against"].cumsum()
    shots_to_date["c_shots_diff"] = g["shot_attempts_differential"].cumsum()

    shots_to_date["c_corsi_for"] = g["corsi_for"].cumsum()
    shots_to_date["c_corsi_against"] = g["corsi_against"].cumsum()
    shots_to_date["c_corsi_diff"] = (
        shots_to_date["c_corsi_for"] - shots_to_date["c_corsi_against"]
    )

    # -------------------------------------------------
    # Games played
    # -------------------------------------------------
    shots_to_date["games_played"] = g.cumcount() + 1

    # -------------------------------------------------
    # Per-game rates
    # -------------------------------------------------
    shots_to_date["shot_attempts_for_per_game"] = np.where(
        shots_to_date["games_played"] > 0,
        shots_to_date["c_shots_for"] / shots_to_date["games_played"],
        0.0
    )

    shots_to_date["shot_attempts_against_per_game"] = np.where(
        shots_to_date["games_played"] > 0,
        shots_to_date["c_shots_against"] / shots_to_date["games_played"],
        0.0
    )

    shots_to_date["shots_diff_per_game"] = np.where(
        shots_to_date["games_played"] > 0,
        shots_to_date["c_shots_diff"] / shots_to_date["games_played"],
        0.0
    )

    shots_to_date["corsi_diff_per_game"] = np.where(
        shots_to_date["games_played"] > 0,
        shots_to_date["c_corsi_diff"] / shots_to_date["games_played"],
        0.0
    )

    return shots_to_date


In [134]:
curr_cumulative_shots = cumulative_shot_attempts(curr_shots_stats)
hist_cumulative_shots = cumulative_shot_attempts(hist_shots_stats)

In [135]:

curr_cumulative_shots.head()

,season_id,game_id,team_id,shot_attempts_for,shot_attempts_against,shot_attempts_differential,corsi_for,corsi_against,corsi_diff,c_shots_for,c_shots_against,c_shots_diff,c_corsi_for,c_corsi_against,c_corsi_diff,games_played,shot_attempts_for_per_game,shot_attempts_against_per_game,shots_diff_per_game,corsi_diff_per_game
26,20252026,2025020014,1,45,45,0,30,33,-3,45,45,0,30,33,-3,1,45.0,45.000000,0.000000,-3.000000
50,20252026,2025020026,1,64,64,0,43,42,1,109,109,0,73,75,-2,2,54.5,54.500000,0.000000,-1.000000
88,20252026,2025020045,1,56,55,1,40,39,1,165,164,1,113,114,-1,3,55.0,54.666667,0.333333,-0.333333
126,20252026,2025020064,1,67,66,1,48,42,6,232,230,2,161,156,5,4,58.0,57.500000,0.500000,1.250000
154,20252026,2025020078,1,51,51,0,36,34,2,283,281,2,197,190,7,5,56.6,56.200000,0.400000,1.400000


### Danger Zones

In [136]:

#This is just identifying the location of shot events to determine their danger zone 

def preprocess_shot_dataframe(df):
    required_cols = {
        "x_coord", "y_coord", "game_id",
        "period_number", "game_in_seconds", 'sortOrder'
    }
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df = df.copy()

    # -----------------------------
    # 1. Boolean danger regions
    # -----------------------------
    hd = (
        ((df["x_coord"] >= 66.5) & (df["x_coord"] <= 89) &
         (df["y_coord"].between(-7, 7)))
        |
        ((df["x_coord"].between(-89, -66.5)) &
         (df["y_coord"].between(-7, 7)))
    )

    md = (
        # Right side
        ((df["x_coord"].between(69, 89)) & df["y_coord"].between(-22, -7))
        |
        ((df["x_coord"].between(54, 69)) & df["y_coord"].between(-22, 22))
        |
        ((df["x_coord"].between(40, 54)) & df["y_coord"].between(-7, 7))
        |
        # Left side mirrored
        ((df["x_coord"].between(-89, -69)) & df["y_coord"].between(-22, -7))
        |
        ((df["x_coord"].between(-69, -54)) & df["y_coord"].between(-22, 22))
        |
        ((df["x_coord"].between(-54, -40)) & df["y_coord"].between(-7, 7))
    )

    # Low danger = not HD or MD
    ld = ~(hd | md)

    # -----------------------------
    # 2. Assign numeric danger level
    # -----------------------------
    df["hd_shot"] = hd.astype(int)
    df["md_shot"] = md.astype(int)
    df["ld_shot"] = ld.astype(int)

    # -----------------------------
    # 3. Timing features
    # -----------------------------
    df = df.sort_values(by=['game_id', 'sortOrder'])
    
    df["game_time"] = (df["period_number"] - 1) * 1200 + df["game_in_seconds"]

    df["time_since_last_shot"] = df.groupby("game_id")["game_time"].diff()

    df["is_rebound"] = (df["time_since_last_shot"] <= 3).astype(int)

    return df


In [137]:
# These are shot attempt events
hist_danger_event = preprocess_shot_dataframe(hist_shot_events)
curr_danger_event = preprocess_shot_dataframe(curr_shot_events)
curr_danger_event.head()



,game_id,team_id,home_team,away_team,type_code,typeDescKey,zone_code,x_coord,y_coord,period_number,game_in_seconds,goalie,sortOrder,season_id,shot_attempt,prev_team_id,hd_shot,md_shot,ld_shot,game_time,time_since_last_shot,is_rebound
2,2025020001,13,13,16,508,blocked-shot,D,-61.0,3.0,1,22,<NA>,12,20252026,1,<NA>,0,1,0,22,NaN,0
7,2025020001,13,13,16,508,blocked-shot,D,-64.0,-16.0,1,73,<NA>,25,20252026,1,13,0,1,0,73,51.0,0
10,2025020001,13,13,16,506,shot-on-goal,O,-58.0,-22.0,1,76,8481519,29,20252026,1,13,0,1,0,76,3.0,1
15,2025020001,13,13,16,506,shot-on-goal,O,-33.0,-19.0,1,95,8481519,35,20252026,1,13,0,0,1,95,19.0,0
18,2025020001,16,13,16,507,missed-shot,O,55.0,-29.0,1,104,8475683,38,20252026,1,13,0,0,1,104,9.0,0


### Scoring Chances

In [138]:
def add_scoring_chances(df):
    """
    Adds scoring chance classifications at the EVENT level.

    Produces:
      - base_scoring_chance_score (quality only)
      - adj_scoring_chance_score (quality + context)
      - scoring_chance (binary, base definition)
      - rush_scoring_chance (contextual)
    """
    d = df.copy()

    # -------------------------------------------------
    # Order events (important for prev_team_id)
    # -------------------------------------------------
    if "sortOrder" in d.columns:
        d = d.sort_values(["game_id", "sortOrder"])

    # -------------------------------------------------
    # Previous team (for AGAINST attribution later)
    # -------------------------------------------------
    d["prev_team_id"] = d.groupby("game_id")["team_id"].shift(1)

    # -------------------------------------------------
    # BASE scoring chance score (NO context)
    # HD=3, MD=2, LD=1
    # -------------------------------------------------
    d["base_scoring_chance_score"] = (
        3 * d["hd_shot"]
        + 2 * d["md_shot"]
        + 1 * d["ld_shot"]
    )

    # -------------------------------------------------
    # BASE binary scoring chance
    # (this is what SCF / SCA should use)
    # -------------------------------------------------
    d["scoring_chance"] = (
        d["base_scoring_chance_score"] >= 2
    ).astype(int)

    # -------------------------------------------------
    # CONTEXTUAL bonuses / penalties
    # -------------------------------------------------
    d["context_bonus"] = (
        (d["is_rebound"] == 1).astype(int)
        + (d["rush_shot"] == 1).astype(int)
    )

    d["blocked_penalty"] = (
        d["type_code"] == 508
    ).astype(int)

    # -------------------------------------------------
    # ADJUSTED scoring chance score
    # (use for weighting, not counting)
    # -------------------------------------------------
    d["adj_scoring_chance_score"] = (
        d["base_scoring_chance_score"]
        + d["context_bonus"]
        - d["blocked_penalty"]
    )

    # -------------------------------------------------
    # Outcome flags
    # -------------------------------------------------
    d["scoring_chance_goal"] = (
        d["type_code"] == 505
    ).astype(int)

    d["shot_on_goal"] = (
        (d["type_code"] == 506).astype(int)
        + d["scoring_chance_goal"]
    )

    # -------------------------------------------------
    # Contextual rush scoring chance (subset)
    # -------------------------------------------------
    d["rush_scoring_chance"] = (
        (d["scoring_chance"] == 1)
        & (d["rush_shot"] == 1)
    ).astype(int)

    return d


#### Combined data

In [139]:
hist_combined_chances_data = hist_shot_events.merge(hist_danger_event[['season_id', 'game_id', 'team_id', 'sortOrder', 'hd_shot', 'md_shot', 'ld_shot', 'is_rebound']], 
                                               on =  ['season_id', 'game_id', 'team_id', 'sortOrder'], how = "left")

hist_combined_chances_data = hist_combined_chances_data.merge(hist_rush_events[['season_id', 'game_id', 'team_id', 'sortOrder', 'rush_shot']], 
                                               on =  ['season_id', 'game_id', 'team_id', 'sortOrder'], how = "left")



In [140]:

curr_combined_chances_data = curr_shot_events.merge(curr_danger_event[['season_id', 'game_id', 'team_id', 'sortOrder', 'hd_shot', 'md_shot', 'ld_shot', 'is_rebound']], 
                                               on =  ['season_id', 'game_id', 'team_id', 'sortOrder'], how = "left")

curr_combined_chances_data = curr_combined_chances_data.merge(curr_rush_events[['season_id', 'game_id', 'team_id', 'sortOrder', 'rush_shot']], 
                                               on =  ['season_id', 'game_id', 'team_id', 'sortOrder'], how = "left")


In [141]:
hist_scoring_chances_event = add_scoring_chances(hist_combined_chances_data)

curr_scoring_chances_event = add_scoring_chances(curr_combined_chances_data)


In [142]:
curr_scoring_chances_event.head()

,game_id,team_id,home_team,away_team,type_code,typeDescKey,zone_code,x_coord,y_coord,period_number,game_in_seconds,goalie,sortOrder,season_id,shot_attempt,prev_team_id,hd_shot,md_shot,ld_shot,is_rebound,rush_shot,base_scoring_chance_score,scoring_chance,context_bonus,blocked_penalty,adj_scoring_chance_score,scoring_chance_goal,shot_on_goal,rush_scoring_chance
0,2025020001,13,13,16,508,blocked-shot,D,-61.0,3.0,1,22,<NA>,12,20252026,1,<NA>,0,1,0,0,NaN,2,1,0,1,1,0,0,0
1,2025020001,13,13,16,508,blocked-shot,D,-64.0,-16.0,1,73,<NA>,25,20252026,1,13,0,1,0,0,1.0,2,1,1,1,2,0,0,1
2,2025020001,13,13,16,506,shot-on-goal,O,-58.0,-22.0,1,76,8481519,29,20252026,1,13,0,1,0,1,NaN,2,1,1,0,3,0,1,0
3,2025020001,13,13,16,506,shot-on-goal,O,-33.0,-19.0,1,95,8481519,35,20252026,1,13,0,0,1,0,NaN,1,0,0,0,1,0,1,0
4,2025020001,16,13,16,507,missed-shot,O,55.0,-29.0,1,104,8475683,38,20252026,1,13,0,0,1,0,NaN,1,0,0,0,1,0,0,0


### Danger Corsi and SCF Stats

In [143]:
def build_danger_corsi_and_scf_stats(df):
    """
    Produces:
        - HDCF, HDCA
        - MDCF, MDCA
        - LDCF, LDCA
        - SCF (scoring chances for)
        - SCSF (scoring-chance shots for)
        - SCGF (scoring-chance goals for)
        - SCA (scoring chances against)

    Requires df already to contain:
        hd_shot, md_shot, ld_shot
        rush_shot, is_rebound
        scoring_chance
        home_team, away_team
    """

    d = df.copy()

    # -----------------------------
    # Assign shooting team correctly
    # -----------------------------
    d["shooting_team_id"] = d["team_id"]

    mask_block = d["type_code"] == 508
    home_block = d["team_id"] == d["home_team"]
    away_block = d["team_id"] == d["away_team"]

    d.loc[mask_block & home_block, "shooting_team_id"] = d["away_team"]
    d.loc[mask_block & away_block, "shooting_team_id"] = d["home_team"]

    # -----------------------------
    # Scoring chance shot + goal flags
    # -----------------------------
    d["shot_on_goal"] = d["type_code"].isin([505, 506]).astype(int)
    d["scoring_chance_goal"] = (d["type_code"] == 505).astype(int)

    # -----------------------------
    # FOR — scoring chances
    # -----------------------------
    scf = (
        d.groupby(["season_id", "game_id", "shooting_team_id"], as_index=False)
         .agg(
             SCF=("scoring_chance", "sum"),
             SCSF=("shot_on_goal", "sum"),
             SCGF=("scoring_chance_goal", "sum")
         )
         .rename(columns={"shooting_team_id": "team_id"})
    )

    # -----------------------------
    # AGAINST — scoring chances
    # -----------------------------
    # Home team allows all away shots; away team allows all home shots
    sca_home = (
        d.groupby(["season_id", "game_id", "home_team"], as_index=False)
         .agg(SCA=("scoring_chance", "sum"))
         .rename(columns={"home_team": "team_id"})
    )
    sca_away = (
        d.groupby(["season_id", "game_id", "away_team"], as_index=False)
         .agg(SCA=("scoring_chance", "sum"))
         .rename(columns={"away_team": "team_id"})
    )

    sca = (
        pd.concat([sca_home, sca_away])
          .groupby(["season_id", "game_id", "team_id"], as_index=False)
          .agg(SCA=("SCA", "sum"))
    )

    # -----------------------------
    # Danger-split Corsi FOR
    # -----------------------------
    d["hd_corsi"] = d["hd_shot"]
    d["md_corsi"] = d["md_shot"]
    d["ld_corsi"] = d["ld_shot"]

    corsi_for = (
        d.groupby(["season_id", "game_id", "team_id"], as_index=False)
         .agg(
             HDCF=("hd_corsi", "sum"),
             MDCF=("md_corsi", "sum"),
             LDCF=("ld_corsi", "sum")
         )
    )

    # -----------------------------
    # Danger-split Corsi AGAINST
    # -----------------------------
    corsi_against = (
        d.groupby(["season_id", "game_id", "shooting_team_id"], as_index=False)
         .agg(
             HDCA=("hd_corsi", "sum"),
             MDCA=("md_corsi", "sum"),
             LDCA=("ld_corsi", "sum")
         )
         .rename(columns={"shooting_team_id": "team_id"})
    )

    # -----------------------------
    # FINAL MERGE
    # -----------------------------
    final = (
        scf.merge(sca, on=["season_id", "game_id", "team_id"], how="left")
           .merge(corsi_for, on=["season_id", "game_id", "team_id"], how="left")
           .merge(corsi_against, on=["season_id", "game_id", "team_id"], how="left")
    )
    
    # -----------------------------
    # Percentages
    # -----------------------------
    final["HDCF_pct"] = final["HDCF"] / (final["HDCF"] + final["HDCA"]).replace(0, np.nan)
    final["HDCA_pct"] = final["HDCA"] / (final["HDCF"] + final["HDCA"]).replace(0, np.nan)

    final["MDCF_pct"] = final["MDCF"] / (final["MDCF"] + final["MDCA"]).replace(0, np.nan)
    final["MDCA_pct"] = final["MDCA"] / (final["MDCF"] + final["MDCA"]).replace(0, np.nan)

    final["LDCF_pct"] = final["LDCF"] / (final["LDCF"] + final["LDCA"]).replace(0, np.nan)
    final["LDCA_pct"] = final["LDCA"] / (final["LDCF"] + final["LDCA"]).replace(0, np.nan)

    return final


In [ ]:
curr_danger_scoring_stats = build_danger_corsi_and_scf_stats(curr_scoring_chances_event)


hist_danger_scoring_stats = build_danger_corsi_and_scf_stats(hist_scoring_chances_event)

In [145]:
hist_danger_scoring_stats.head()

,season_id,game_id,team_id,SCF,SCSF,SCGF,SCA,HDCF,MDCF,LDCF,HDCA,MDCA,LDCA,HDCF_pct,HDCA_pct,MDCF_pct,MDCA_pct,LDCF_pct,LDCA_pct
0,20132014,2013020001,8,34,37,3,65.0,15,20,38,16,22,34,0.483871,0.516129,0.476190,0.523810,0.527778,0.472222
1,20132014,2013020001,10,31,38,4,65.0,7,27,28,6,25,32,0.538462,0.461538,0.519231,0.480769,0.466667,0.533333
2,20132014,2013020002,15,30,32,4,64.0,13,25,22,11,21,24,0.541667,0.458333,0.543478,0.456522,0.478261,0.521739
3,20132014,2013020002,16,34,35,6,64.0,13,18,26,15,22,24,0.464286,0.535714,0.450000,0.550000,0.520000,0.480000
4,20132014,2013020003,22,38,38,4,76.0,15,29,28,11,28,24,0.576923,0.423077,0.508772,0.491228,0.538462,0.461538


In [146]:
hist_shots_stats.head()

,season_id,game_id,team_id,shot_attempts_for,shot_attempts_against,shot_attempts_differential,corsi_for,corsi_against,corsi_diff
0,20132014,2013020001,8,70,69,1,53.0,53.0,0.0
1,20132014,2013020001,10,61,61,0,45.0,44.0,1.0
2,20132014,2013020002,15,57,57,0,39.0,39.0,0.0
3,20132014,2013020002,16,55,54,1,38.0,38.0,0.0
4,20132014,2013020003,22,71,70,1,48.0,49.0,-1.0


#### Cumulative Scoring Chances

In [147]:
def cumulative_scoring_chances(danger_scoring: pd.DataFrame,
                               corsi_stats: pd.DataFrame) -> pd.DataFrame:
    scoring = danger_scoring.copy()
    shots = corsi_stats.copy()

    # -------------------------------------------------
    # Sort both dataframes consistently
    # -------------------------------------------------
    sort_cols = (
        ["season_id", "team_id", "game_date", "game_id"]
        if "game_date" in scoring.columns
        else ["season_id", "team_id", "game_id"]
    )

    scoring = scoring.sort_values(sort_cols)
    shots = shots.sort_values(sort_cols)

    # -------------------------------------------------
    # Grouping
    # -------------------------------------------------
    g_sc = scoring.groupby(["season_id", "team_id"])
    g_cf = shots.groupby(["season_id", "team_id"])

    # -------------------------------------------------
    # Cumulative scoring-chance metrics
    # -------------------------------------------------
    scoring["c_SCF"] = g_sc["SCF"].cumsum()
    scoring["c_SCSF"] = g_sc['SCSF'].cumsum()
    scoring['c_SCGF'] = g_sc['SCGF'].cumsum()
    scoring['c_SCA'] = g_sc['SCA'].cumsum()
    
    scoring["games_played"] = g_sc.cumcount() + 1

    scoring["SCF_pg"] = scoring["c_SCF"] / scoring["games_played"]
    scoring["SCA_pg"] = scoring["c_SCA"] / scoring["games_played"]
    
    
    scoring["c_HDCF"] = g_sc["HDCF"].cumsum()
    scoring["c_HDCA"] = g_sc["HDCA"].cumsum()
    scoring["c_MDCF"] = g_sc["MDCF"].cumsum()
    scoring["c_MDCA"] = g_sc["MDCA"].cumsum()
    scoring["c_LDCF"] = g_sc["LDCF"].cumsum()
    scoring["c_LDCA"] = g_sc["LDCA"].cumsum()


    # -------------------------------------------------
    # Cumulative Corsi metrics
    # -------------------------------------------------
    shots["c_corsi_for"] = g_cf["corsi_for"].cumsum()
    shots["c_corsi_against"] = g_cf["corsi_against"].cumsum()
    shots["c_corsi_diff"] = (
        shots["c_corsi_for"] - shots["c_corsi_against"]
    )

    shots["games_played"] = g_cf.cumcount() + 1
    shots["corsi_diff_per_game"] = (
        shots["c_corsi_diff"] / shots["games_played"]
    )




    # -------------------------------------------------
    # Merge back together
    # -------------------------------------------------
    merge_cols = ["season_id", "team_id", "game_id"]
    if "game_date" in scoring.columns and "game_date" in shots.columns:
        merge_cols.append("game_date")

    out = scoring.merge(
        shots[
            merge_cols + [
                "c_corsi_for",
                "c_corsi_against",
                "c_corsi_diff",
                "corsi_diff_per_game",

            ]
        ],
        on=merge_cols,
        how="left",
    )
    out = out.drop(columns=['games_played'])

    return out


In [148]:
hist_cumulative_scoring_chances =cumulative_scoring_chances(hist_danger_scoring_stats, hist_shots_stats)
curr_cumulative_scoring_chances =cumulative_scoring_chances(curr_danger_scoring_stats,curr_shots_stats)

curr_cumulative_scoring_chances.head()

,season_id,game_id,team_id,SCF,SCSF,SCGF,SCA,HDCF,MDCF,LDCF,HDCA,MDCA,LDCA,HDCF_pct,HDCA_pct,MDCF_pct,MDCA_pct,LDCF_pct,LDCA_pct,c_SCF,c_SCSF,c_SCGF,c_SCA,SCF_pg,SCA_pg,c_HDCF,c_HDCA,c_MDCF,c_MDCA,c_LDCF,c_LDCA,c_corsi_for,c_corsi_against,c_corsi_diff,corsi_diff_per_game
0,20252026,2025020014,1,25,22,3,64,7,15,25,12,15,26,0.368421,0.631579,0.500000,0.500000,0.490196,0.509804,25,22,3,64,25.000000,64.000000,7,12,15,15,25,26,30,33,-3,-3.000000
1,20252026,2025020026,1,42,31,5,72,18,24,25,22,23,30,0.450000,0.550000,0.510638,0.489362,0.454545,0.545455,67,53,8,136,33.500000,68.000000,25,34,39,38,50,56,73,75,-2,-1.000000
2,20252026,2025020045,1,31,28,3,64,14,15,29,15,17,33,0.482759,0.517241,0.468750,0.531250,0.467742,0.532258,98,81,11,200,32.666667,66.666667,39,49,54,55,79,89,113,114,-1,-0.333333
3,20252026,2025020064,1,36,33,3,61,16,16,36,16,21,35,0.500000,0.500000,0.432432,0.567568,0.507042,0.492958,134,114,14,261,33.500000,65.250000,55,65,70,76,115,124,161,156,5,1.250000
4,20252026,2025020078,1,33,23,5,58,17,14,23,20,15,21,0.459459,0.540541,0.482759,0.517241,0.522727,0.477273,167,137,19,319,33.400000,63.800000,72,85,84,91,138,145,197,190,7,1.400000


### Shooting Percentage (shots on goal)

In [149]:
def build_team_shooting_stats(df):
    df = df.copy()

    df["goal"]  = df['scoring_chance_goal']
    df["sog"]   = df['shot_on_goal']


    df["hd_sog"] = ((df["sog"] == 1) & (df["hd_shot"] == 1)).astype(int)
    df["md_sog"] = ((df["sog"] == 1) & (df["md_shot"] == 1)).astype(int)
    df["ld_sog"] = ((df["sog"] == 1) & (df["ld_shot"] == 1)).astype(int)

    df["hd_goal"] = ((df["goal"] == 1) & (df["hd_shot"] == 1)).astype(int)
    df["md_goal"] = ((df["goal"] == 1) & (df["md_shot"] == 1)).astype(int)
    df["ld_goal"] = ((df["goal"] == 1) & (df["ld_shot"] == 1)).astype(int)

    opp_map = (
        df[["game_id","team_id"]].drop_duplicates()
          .groupby("game_id")["team_id"].apply(list)
    )
    opponent = {
        gid: ({t[0]:t[1], t[1]:t[0]} if len(t)==2 else {})
        for gid, t in opp_map.items()
    }
    df["opponent_team_id"] = df.apply(
        lambda r: opponent.get(r["game_id"],{}).get(r["team_id"]),
        axis=1
    )

    stats_for = (
        df.groupby(["season_id", "game_id","team_id"], as_index=False)
          .agg(
              shots_for=("sog","sum"),
              goals_for=("goal","sum"),
              hd_sog=("hd_sog","sum"),
              md_sog=("md_sog","sum"),
              ld_sog=("ld_sog","sum"),
              hd_goals=("hd_goal","sum"),
              md_goals=("md_goal","sum"),
              ld_goals=("ld_goal","sum"),
          )
    )
    stats_for["shot_pct"] = 100 * stats_for["goals_for"] / stats_for["shots_for"].replace(0,np.nan)

    stats_against = (
        df.groupby(["season_id", "game_id","opponent_team_id"], as_index=False)
          .agg(
              shots_against=("sog","sum"),
              goals_against=("goal","sum"),
              hd_against=("hd_sog","sum"),
              md_against=("md_sog","sum"),
              ld_against=("ld_sog","sum"),
          )
          .rename(columns={"opponent_team_id":"team_id"})
    )

    stats = stats_for.merge(stats_against, on=["season_id", "game_id","team_id"], how="left")

    stats["sv_pct"] = 100 - (100 * stats["goals_against"] / stats["shots_against"].replace(0,np.nan))
    stats["pdo"] = stats["shot_pct"] + stats["sv_pct"]

    return stats


In [150]:
curr_sog_stats = build_team_shooting_stats(curr_scoring_chances_event)
hist_sog_stats = build_team_shooting_stats(hist_scoring_chances_event)

In [151]:
curr_sog_stats.head()

,season_id,game_id,team_id,shots_for,goals_for,hd_sog,md_sog,ld_sog,hd_goals,md_goals,ld_goals,shot_pct,shots_against,goals_against,hd_against,md_against,ld_against,sv_pct,pdo
0,20252026,2025020001,13,37,3,10,11,16,2,0,1,8.108108,19,2,3,5,12,89.473684,97.581792
1,20252026,2025020001,16,19,2,3,5,12,0,2,0,10.526316,37,3,10,11,16,91.891892,102.418208
2,20252026,2025020002,3,25,0,4,8,14,0,0,0,0.000000,31,3,10,7,14,90.322581,90.322581
3,20252026,2025020002,5,31,3,10,7,14,2,0,1,9.677419,25,0,4,8,14,100.000000,109.677419
4,20252026,2025020003,21,23,4,7,3,14,1,1,2,17.391304,25,1,7,6,14,96.000000,113.391304


### Cumulative SOG 

In [152]:
def cumulative_sog(df: pd.DataFrame) -> pd.DataFrame:

    sog = df.copy()

    # -------------------------------------------------
    # Sort BEFORE cumulative math
    # -------------------------------------------------
    sort_cols = (
        ["season_id", "team_id", "game_date", "game_id"]
        if "game_date" in sog.columns
        else ["season_id", "team_id", "game_id"]
    )
    sog = sog.sort_values(sort_cols)

    # -------------------------------------------------
    # Group by season + team
    # -------------------------------------------------
    g = sog.groupby(["season_id", "team_id"])

    # -------------------------------------------------
    # Games played
    # -------------------------------------------------
    sog["games_played"] = g.cumcount() + 1

    # -------------------------------------------------
    # Cumulative FOR
    # -------------------------------------------------
    sog["c_shots_for"]  = g["shots_for"].cumsum()
    sog["c_goals_for"]  = g["goals_for"].cumsum()

    sog["c_hd_sog"] = g["hd_sog"].cumsum()
    sog["c_md_sog"] = g["md_sog"].cumsum()
    sog["c_ld_sog"] = g["ld_sog"].cumsum()

    sog["c_hd_goals"] = g["hd_goals"].cumsum()
    sog["c_md_goals"] = g["md_goals"].cumsum()
    sog["c_ld_goals"] = g["ld_goals"].cumsum()

    # -------------------------------------------------
    # Cumulative AGAINST
    # -------------------------------------------------
    sog["c_shots_against"]  = g["shots_against"].cumsum()
    sog["c_goals_against"]  = g["goals_against"].cumsum()

    sog["c_hd_against"] = g["hd_against"].cumsum()
    sog["c_md_against"] = g["md_against"].cumsum()
    sog["c_ld_against"] = g["ld_against"].cumsum()

    # -------------------------------------------------
    # Recompute percentages (DO NOT cumsum these)
    # -------------------------------------------------
    sog["c_shot_pct"] = np.where(
        sog["c_shots_for"] > 0,
        sog["c_goals_for"] / sog["c_shots_for"],
        0.0
    )

    sog["c_sv_pct"] = np.where(
        sog["c_shots_against"] > 0,
        1 - (sog["c_goals_against"] / sog["c_shots_against"]),
        0.0
    )

    sog["c_pdo"] = (sog["c_shot_pct"] + sog["c_sv_pct"]) * 100
    sog=sog.drop(columns=['games_played'])

    return sog

    


In [153]:

hist_cumulative_sog = cumulative_sog(hist_sog_stats)
curr_cumulative_sog = cumulative_sog(curr_sog_stats)

curr_cumulative_sog.head()

,season_id,game_id,team_id,shots_for,goals_for,hd_sog,md_sog,ld_sog,hd_goals,md_goals,ld_goals,shot_pct,shots_against,goals_against,hd_against,md_against,ld_against,sv_pct,pdo,c_shots_for,c_goals_for,c_hd_sog,c_md_sog,c_ld_sog,c_hd_goals,c_md_goals,c_ld_goals,c_shots_against,c_goals_against,c_hd_against,c_md_against,c_ld_against,c_shot_pct,c_sv_pct,c_pdo
26,20252026,2025020014,1,22,3,3,5,15,1,2,1,13.636364,33,6,5,12,16,81.818182,95.454545,22,3,3,5,15,1,2,1,33,6,5,12,16,0.136364,0.818182,95.454545
50,20252026,2025020026,1,31,5,11,11,11,4,1,1,16.129032,17,3,4,5,8,82.352941,98.481973,53,8,14,16,26,5,3,2,50,9,9,17,24,0.150943,0.820000,97.094340
88,20252026,2025020045,1,28,3,4,7,17,0,1,2,10.714286,33,2,5,10,18,93.939394,104.653680,81,11,18,23,43,5,4,4,83,11,14,27,42,0.135802,0.867470,100.327235
126,20252026,2025020064,1,33,3,6,9,19,1,3,0,9.090909,22,1,5,6,12,95.454545,104.545455,114,14,24,32,62,6,7,4,105,12,19,33,54,0.122807,0.885714,100.852130
154,20252026,2025020078,1,23,5,9,6,9,2,1,2,21.739130,30,3,5,6,19,90.000000,111.739130,137,19,33,38,71,8,8,6,135,15,24,39,73,0.138686,0.888889,102.757502


## Strength of Schedule - need to update this re: off/fut games

In [154]:
def strength_of_schedule(df):
    sos = df.copy()

    games_sos = []

    # -------------------------------------------------
    # Build two rows per game (HOME + AWAY)
    # -------------------------------------------------
    for _, row in sos.iterrows():

        # AWAY team row
        games_sos.append({
            "season_id": row["season_id"],
            "game_id": int(row["game_id"]),
            "team_id": int(row["away_team"]),
            "tri_code": row["away_tri_code"],
            "opponent_id": int(row["home_team"]),
            "opponent": row["home_tri_code"],
            "goals_for": row["awayTeam.score"],
            "goals_against": row["homeTeam.score"],
        })

        # HOME team row
        games_sos.append({
            "season_id": row["season_id"],
            "game_id": int(row["game_id"]),
            "team_id": int(row["home_team"]),
            "tri_code": row["home_tri_code"],
            "opponent_id": int(row["away_team"]),
            "opponent": row["away_tri_code"],
            "goals_for": row["homeTeam.score"],
            "goals_against": row["awayTeam.score"],
        })

    games_sos = pd.DataFrame(games_sos)

    # -------------------------------------------------
    # Sort + win calculation
    # -------------------------------------------------
    games_sos = games_sos.sort_values(
        ["season_id", "team_id", "tri_code", "game_id"]
    )

    games_sos["win"] = (
        games_sos["goals_for"] > games_sos["goals_against"]
    ).astype(int)

    # -------------------------------------------------
    # Wins BEFORE (leave NaN for first game)
    # -------------------------------------------------
    games_sos["wins_before"] = (
        games_sos
        .groupby(["season_id", "team_id", "tri_code"])["win"]
        .transform(lambda s: s.cumsum().shift(1))
        .astype("Int64")
    )


    # -------------------------------------------------
    # Games BEFORE / played
    # -------------------------------------------------
    games_sos["games_before"] = (
        games_sos
        .groupby(["season_id", "team_id", "tri_code"])
        .cumcount()
    )

    games_sos["games_played"] = games_sos["games_before"] + 1

    # -------------------------------------------------
    # Win% BEFORE (NaN if no prior games)
    # -------------------------------------------------
    games_sos["win_pct_before"] = np.where(
        games_sos["games_before"] > 0,
        games_sos["wins_before"] / games_sos["games_before"],
        np.nan
    )

    # -------------------------------------------------
    # Deduplicate opponent win% BEFORE merge
    # -------------------------------------------------
    opp_win = (
        games_sos[["season_id", "tri_code", "game_id", "win_pct_before"]]
        .drop_duplicates(subset=["season_id", "tri_code", "game_id"])
    )

    games_sos = games_sos.merge(
        opp_win.rename(columns={
            "tri_code": "opponent",
            "win_pct_before": "opponent_win_pct_before"
        }),
        on=["season_id", "opponent", "game_id"],
        how="left"
    )

    # -------------------------------------------------
    # SOS — PAST (NaN until at least one prior game)
    # -------------------------------------------------
    games_sos["sos_past"] = (
        games_sos
        .groupby(["season_id", "tri_code"])["opponent_win_pct_before"]
        .expanding()
        .mean()
        .reset_index(level=[0, 1], drop=True)
    )

    # -------------------------------------------------
    # SOS — FUTURE (reverse expanding, valid from game 1)
    # -------------------------------------------------
    games_sos["sos_future"] = (
        games_sos.iloc[::-1]
        .groupby(["season_id", "team_id", "tri_code"])["opponent_win_pct_before"]
        .expanding()
        .mean()
        .reset_index(level=[0, 1, 2], drop=True)
        .iloc[::-1]
    )

    # -------------------------------------------------
    # Final cleanup
    # -------------------------------------------------
    games_sos = (
        games_sos
        .drop(columns=['opponent_id', 'opponent', 'goals_for', 'goals_against', 'opponent_win_pct_before'])
        .sort_values(by=["season_id", "game_id"])
    )

    return games_sos


In [155]:
hist_games_sos = strength_of_schedule(history_game_summary)
curr_games_sos = strength_of_schedule(game_summary)


## Goalies

In [156]:
history_goalies = con.execute("""
                              SELECT *
                              FROM all_goalie_games
                              WHERE season_id !=20252026
                              ORDER BY goalie_name""").df()

history_goalies.head()

,game_id,games_played,games_started,goalie_name,player_id,tri_code,toi,wins,losses,ot_losses,gaa,saves,shutouts,save_pct,shots_against,goals_against,season_id,team_id
0,2016020802,1,0,Aaron Dell,8477180,SJS,2287,0,1,0,3.14822,18,0,0.90000,20,2,20162017,28
1,2018020239,1,1,Aaron Dell,8477180,SJS,3600,0,1,0,4.00000,27,0,0.87096,31,4,20182019,28
2,2019021079,1,1,Aaron Dell,8477180,SJS,3588,0,1,0,6.02006,39,0,0.86666,45,6,20192020,28
3,2021020211,1,1,Aaron Dell,8477180,BUF,3588,0,1,0,5.01672,21,0,0.80769,26,5,20212022,7
4,2019020389,1,1,Aaron Dell,8477180,SJS,3585,0,1,0,4.01673,21,0,0.84000,25,4,20192020,28


In [157]:
goalies = con.execute("""
                      SELECT *
                      FROM all_goalie_games
                      WHERE season_id =20252026
                      ORDER BY goalie_name""").df()

goalies.head()

,game_id,games_played,games_started,goalie_name,player_id,tri_code,toi,wins,losses,ot_losses,gaa,saves,shutouts,save_pct,shots_against,goals_against,season_id,team_id
0,2025020053,1,1,Adin Hill,8478499,VGK,1200,0,0,0,6.00000,8,0,0.80000,10,2,20252026,54
1,2025020087,1,1,Adin Hill,8478499,VGK,3598,1,0,0,1.00055,26,0,0.96296,27,1,20252026,54
2,2025020007,1,1,Adin Hill,8478499,VGK,3893,0,0,1,4.62368,21,0,0.80769,26,5,20252026,54
3,2025020097,1,1,Adin Hill,8478499,VGK,594,0,0,0,0.00000,4,0,1.00000,4,0,20252026,54
4,2025020037,1,1,Adin Hill,8478499,VGK,3896,0,0,1,1.84804,20,0,0.90909,22,2,20252026,54


In [158]:
def build_team_goalie_stats(goalies_df, teams_df):

    # -----------------------------------------------------
    # Helpers
    # -----------------------------------------------------
    def safe_weighted_mean(values, weights):
        weights = weights.replace(0, np.nan)
        num = (values * weights).sum()
        den = weights.sum()
        return num / den if den > 0 else np.nan

    g = goalies_df.copy()

    # -----------------------------------------------------
    # 1. Individual goalie performance per game
    # -----------------------------------------------------
    goalies_indiv = (
        g.groupby(["goalie_name","season_id", "tri_code", "game_id"])
         .agg(
             i_savePct=("save_pct",
                        lambda x: safe_weighted_mean(
                            x, g.loc[x.index, "shots_against"]
                        )),
             i_GAA=("gaa",
                    lambda x: safe_weighted_mean(
                        x, g.loc[x.index, "toi"]
                    )),
             i_gamesPlayed=("games_played", "sum"),
             i_gamesStarted=("games_started", "sum"),
             i_wins=("wins", "sum"),
             i_losses=("losses", "sum"),
             i_otlosses=("ot_losses", "sum"),
             i_shutouts=("shutouts", "sum"),
             i_saves=("saves", "sum"),
             i_shotsAgainst=("shots_against", "sum"),
             i_timeOnIce=("toi", "sum"),
         )
         .reset_index()
    )

    # -----------------------------------------------------
    # 2. Aggregate to team-level for each game
    # -----------------------------------------------------
    game_goalie_stats = (
        goalies_indiv.groupby(["season_id","game_id", "tri_code"])
        .agg(
            game_savePct=("i_savePct",
                          lambda x: safe_weighted_mean(
                              x, goalies_indiv.loc[x.index, "i_shotsAgainst"]
                          )),
            game_GAA=("i_GAA",
                      lambda x: safe_weighted_mean(
                          x, goalies_indiv.loc[x.index, "i_timeOnIce"]
                      )),
            game_wins=("i_wins", "sum"),
            game_losses=("i_losses", "sum"),
            game_shutouts=("i_shutouts", "sum"),
            game_saves=("i_saves", "sum"),
            game_shotsAgainst=("i_shotsAgainst", "sum"),
        )
        .reset_index()
    )

    # -----------------------------------------------------
    # 3. Output formatting
    # -----------------------------------------------------
    output = game_goalie_stats[["season_id", "game_id", "tri_code", "game_savePct", "game_GAA" ]]
    output = output.merge(teams_df, on="tri_code", how="left")

    return output


In [159]:
curr_cumulative_goalie_stats = build_team_goalie_stats(goalies, teams)
hist_cumulative_goalie_stats = build_team_goalie_stats(history_goalies, teams)


In [160]:
curr_cumulative_goalie_stats.head()

,season_id,game_id,tri_code,game_savePct,game_GAA,team_id
0,20252026,2025020001,CHI,0.91891,3.04740,16
1,20252026,2025020001,FLA,0.89473,2.00000,13
2,20252026,2025020002,NYR,0.96551,1.00642,3
3,20252026,2025020002,PIT,1.00000,0.00000,5
4,20252026,2025020003,COL,0.96000,1.00194,21


## Special Teams

In [161]:
history_pk = con.execute("""
                         SELECT all_pk.*, all_teams.tri_code
                         FROM all_pk
                         join all_teams 
                         on all_teams.team_id = all_pk.team_id
                         WHERE season_id != 20252026
                         ORDER BY game_id""").df()
history_pk.head()

,season_id,game_id,team_id,game_date,games_played,home_road,losses,opponent,ot_losses,pk_net_pct,pk_pct,pk_net_g,pk_net_gpg,pk_toi_pg,points,points_pct,pp_ga,pp_ga_pg,sh_gf,sh_gf_pg,team_name,ties,times_sh,times_sh_pg,wins,tri_code
0,20132014,2013020001,10,2013-10-01,1,R,0,MTL,0,1.250000,1.00000,1,1.0,405.0,2,1.0,0,0.0,1,1.0,Toronto Maple Leafs,None,4,4.0,1,TOR
1,20132014,2013020001,8,2013-10-01,1,H,1,TOR,0,0.857142,0.85714,-1,-1.0,700.0,0,0.0,1,1.0,0,0.0,Montréal Canadiens,None,7,7.0,0,MTL
2,20132014,2013020002,16,2013-10-01,1,H,0,WSH,0,0.500000,0.50000,-3,-3.0,388.0,2,1.0,3,3.0,0,0.0,Chicago Blackhawks,None,6,6.0,1,CHI
3,20132014,2013020002,15,2013-10-01,1,R,1,CHI,0,0.750000,0.75000,-1,-1.0,387.0,0,0.0,1,1.0,0,0.0,Washington Capitals,None,4,4.0,0,WSH
4,20132014,2013020003,22,2013-10-01,1,H,1,WPG,0,0.750000,0.75000,-1,-1.0,400.0,0,0.0,1,1.0,0,0.0,Edmonton Oilers,None,4,4.0,0,EDM


In [162]:
current_pk = con.execute(""" 
                         SELECT  all_pk.*, all_teams.tri_code
                         FROM all_pk
                         join all_teams 
                         on all_teams.team_id = all_pk.team_id
                         WHERE season_id = 20252026
                         ORDER BY game_id""").df()
current_pk.head()

,season_id,game_id,team_id,game_date,games_played,home_road,losses,opponent,ot_losses,pk_net_pct,pk_pct,pk_net_g,pk_net_gpg,pk_toi_pg,points,points_pct,pp_ga,pp_ga_pg,sh_gf,sh_gf_pg,team_name,ties,times_sh,times_sh_pg,wins,tri_code
0,20252026,2025020001,13,2025-10-07,1,H,0,CHI,0,1.00,1.00,0,0.0,360.0,2,1.0,0,0.0,0,0.0,Florida Panthers,None,3,3.0,1,FLA
1,20252026,2025020001,16,2025-10-07,1,R,1,FLA,0,0.50,0.50,-1,-1.0,222.0,0,0.0,1,1.0,0,0.0,Chicago Blackhawks,None,2,2.0,0,CHI
2,20252026,2025020002,3,2025-10-07,1,H,1,PIT,0,1.00,1.00,0,0.0,120.0,0,0.0,0,0.0,0,0.0,New York Rangers,None,1,1.0,0,NYR
3,20252026,2025020002,5,2025-10-07,1,R,0,NYR,0,1.00,1.00,0,0.0,240.0,2,1.0,0,0.0,0,0.0,Pittsburgh Penguins,None,2,2.0,1,PIT
4,20252026,2025020003,21,2025-10-07,1,R,0,LAK,0,0.75,0.75,-1,-1.0,438.0,2,1.0,1,1.0,0,0.0,Colorado Avalanche,None,4,4.0,1,COL


In [163]:
history_pp = con.execute(""" 
                         SELECT  all_pp.*, all_teams.tri_code
                         FROM all_pp
                        join all_teams 
                         on all_teams.team_id = all_pp.team_id
                         WHERE season_id != 20252026
                         ORDER BY game_id""").df()
history_pp.head()

,season_id,game_id,team_id,game_date,games_played,home_road,losses,opponent,ot_losses,point_pct,points,pp_gf,pp_net_pct,pp_pct,pp_gpg,pp_net_g,pp_net_gpg,pp_opportunities,pp_opp_pg,pp_toi,sh_ga,sh_ga_pg,team_name,ties,wins,tri_code
0,20132014,2013020001,10,2013-10-01,1,R,0,MTL,0,1.0,2,1,0.142857,0.14285,1.0,1,1.0,7,7.0,700.0,0,0.0,Toronto Maple Leafs,None,1,TOR
1,20132014,2013020001,8,2013-10-01,1,H,1,TOR,0,0.0,0,0,-0.250000,0.00000,0.0,-1,-1.0,4,4.0,405.0,1,1.0,Montréal Canadiens,None,0,MTL
2,20132014,2013020002,15,2013-10-01,1,R,1,CHI,0,0.0,0,3,0.500000,0.50000,3.0,3,3.0,6,6.0,388.0,0,0.0,Washington Capitals,None,0,WSH
3,20132014,2013020002,16,2013-10-01,1,H,0,WSH,0,1.0,2,1,0.250000,0.25000,1.0,1,1.0,4,4.0,387.0,0,0.0,Chicago Blackhawks,None,1,CHI
4,20132014,2013020003,22,2013-10-01,1,H,1,WPG,0,0.0,0,2,0.400000,0.40000,2.0,2,2.0,5,5.0,399.0,0,0.0,Edmonton Oilers,None,0,EDM


In [164]:
current_pp = con.execute(""" 
                         SELECT all_pp.*, all_teams.tri_code
                         FROM all_pp
                         join all_teams 
                         on all_teams.team_id = all_pp.team_id
                         WHERE season_id = 20252026
                         ORDER BY game_id""").df()
current_pp.head()

,season_id,game_id,team_id,game_date,games_played,home_road,losses,opponent,ot_losses,point_pct,points,pp_gf,pp_net_pct,pp_pct,pp_gpg,pp_net_g,pp_net_gpg,pp_opportunities,pp_opp_pg,pp_toi,sh_ga,sh_ga_pg,team_name,ties,wins,tri_code
0,20252026,2025020001,16,2025-10-07,1,R,1,FLA,0,0.0,0,0,0.000000,0.00000,0.0,0,0.0,3,3.0,360.0,0,0.0,Chicago Blackhawks,None,0,CHI
1,20252026,2025020001,13,2025-10-07,1,H,0,CHI,0,1.0,2,1,0.500000,0.50000,1.0,1,1.0,2,2.0,222.0,0,0.0,Florida Panthers,None,1,FLA
2,20252026,2025020002,5,2025-10-07,1,R,0,NYR,0,1.0,2,0,0.000000,0.00000,0.0,0,0.0,1,1.0,120.0,0,0.0,Pittsburgh Penguins,None,1,PIT
3,20252026,2025020002,3,2025-10-07,1,H,1,PIT,0,0.0,0,0,0.000000,0.00000,0.0,0,0.0,2,2.0,240.0,0,0.0,New York Rangers,None,0,NYR
4,20252026,2025020003,21,2025-10-07,1,R,0,LAK,0,1.0,2,1,0.166666,0.16666,1.0,1,1.0,6,6.0,605.0,0,0.0,Colorado Avalanche,None,1,COL


In [165]:
def build_special_teams_stats(pp_df, pk_df):
    import pandas as pd
    import numpy as np

    pp = pp_df.copy()
    pk = pk_df.copy()

    # ------------------------------------------------------------------
    # 1. Clean + order
    # ------------------------------------------------------------------
    pp = pp.sort_values(["season_id", "team_id", "game_date"])
    pk = pk.sort_values(["season_id", "team_id", "game_date"])

    pp["games_played"] = pp.groupby(["season_id", "team_id"]).cumcount() + 1

    # ------------------------------------------------------------------
    # 2. Merge PP + PK at game level
    # ------------------------------------------------------------------
    merged = pp.merge(
        pk,
        on=["season_id", "game_id", "team_id", "game_date"],
        how="inner",
        suffixes=("_pp", "_pk")
    )

    # ==================================================================
    # ========================= POWER PLAY ==============================
    # ==================================================================
    pp_rank = pp[
        [
            "season_id",
            "game_date",
            "team_id",
            "tri_code",
            "games_played",
            "pp_net_g",
            "pp_opportunities",
        ]
    ].copy()

    pp_rank["c_pp_net_g"] = (
        pp_rank.groupby(["season_id", "team_id"])["pp_net_g"].cumsum()
    )

    pp_rank["c_pp_opps"] = (
        pp_rank.groupby(["season_id", "team_id"])["pp_opportunities"].cumsum()
    )

    pp_rank["net_pp_pct"] = np.where(
        pp_rank["c_pp_opps"] > 0,
        pp_rank["c_pp_net_g"] / pp_rank["c_pp_opps"],
        0
    )

    # ==================================================================
    # ======================== PENALTY KILL =============================
    # ==================================================================
    pk_rank = pk[
        [
            "season_id",
            "game_date",
            "team_id",
            "pp_ga",
            "times_sh",
        ]
    ].copy()

    pk_rank["c_pp_ga"] = (
        pk_rank.groupby(["season_id", "team_id"])["pp_ga"].cumsum()
    )

    pk_rank["c_times_sh"] = (
        pk_rank.groupby(["season_id", "team_id"])["times_sh"].cumsum()
    )

    pk_rank["net_pk_pct"] = np.where(
        pk_rank["c_times_sh"] > 0,
        pk_rank["c_pp_ga"] / pk_rank["c_times_sh"],
        0
    )

    # ------------------------------------------------------------------
    # 3. Build full season-date-team grid
    # ------------------------------------------------------------------
    all_dates = (
        pp_rank.groupby("season_id")["game_date"]
        .unique()
        .explode()
        .unique()
    )

    teams = pp_rank["team_id"].unique()

    grid = pd.MultiIndex.from_product(
        [pp_rank["season_id"].unique(), all_dates, teams],
        names=["season_id", "game_date", "team_id"]
    )

    # ------------------------------------------------------------------
    # 4. Reindex + forward fill (cumulative only)
    # ------------------------------------------------------------------
    pp_full = (
        pp_rank
        .set_index(["season_id", "game_date", "team_id"])
        .reindex(grid)
        .sort_index()
    )

    pk_full = (
        pk_rank
        .set_index(["season_id", "game_date", "team_id"])
        .reindex(grid)
        .sort_index()
    )

    for col in ["c_pp_net_g", "c_pp_opps", "net_pp_pct"]:
        pp_full[col] = pp_full.groupby(["season_id", "team_id"])[col].ffill()

    for col in ["c_pp_ga", "c_times_sh", "net_pk_pct"]:
        pk_full[col] = pk_full.groupby(["season_id", "team_id"])[col].ffill()

    # ------------------------------------------------------------------
    # 5. Percent cleanup + daily ranks
    # ------------------------------------------------------------------
    pp_full["net_pp_pct"] = (pp_full["net_pp_pct"] * 100).round(1).fillna(0)
    pk_full["net_pk_pct"] = (pk_full["net_pk_pct"] * 100).round(1).fillna(0)

    pp_full["ppng_rank"] = (
        pp_full.groupby(["season_id", "game_date"])["net_pp_pct"]
        .rank(method="dense", ascending=False)
        .astype(int)
    )

    pk_full["pk_rank"] = (
        pk_full.groupby(["season_id", "game_date"])["net_pk_pct"]
        .rank(method="dense", ascending=True)
        .astype(int)
    )

    # ------------------------------------------------------------------
    # 6. Merge cumulative stats + ranks back to game rows
    # ------------------------------------------------------------------
    pp_features = pp_full.reset_index()[
        ["season_id", "game_date", "team_id",
         "c_pp_opps", "c_pp_net_g", "net_pp_pct", "ppng_rank"]
    ]

    pk_features = pk_full.reset_index()[
        ["season_id", "game_date", "team_id",
         "c_pp_ga", "c_times_sh", "net_pk_pct", "pk_rank"]
    ]

    special_teams = (
        merged
        .merge(pp_features, on=["season_id", "game_date", "team_id"], how="left")
        .merge(pk_features, on=["season_id", "game_date", "team_id"], how="left")
    )

    # ------------------------------------------------------------------
    # 7. Drop unused columns
    # ------------------------------------------------------------------
    special_teams = special_teams.drop(
        columns=[
            'home_road_pp', 'losses_pp', 'opponent_pp', 'ot_losses_pp',
            'point_pct', 'points_pp', 'team_name_pp', 'ties_pp', 'wins_pp',
            'games_played_pk', 'home_road_pk', 'losses_pk', 'opponent_pk',
            'ot_losses_pk', 'points_pk', 'points_pct', 'team_name_pk',
            'ties_pk', 'wins_pk', 'tri_code_pk', 'games_played_pp'
        ],
        errors="ignore"
    )

    # Rename for consistency
    special_teams = special_teams.rename(
        columns={"pp_opportunities": "pp_opp", "tri_code_pp":"tri_code"}
    )

    # ------------------------------------------------------------------
    # 8. Column ordering
    # ------------------------------------------------------------------
    col_order = [
        'season_id', 'game_id', 'team_id', 'tri_code',

        # PP stats
        'pp_opp', 'pp_opp_pg', 'pp_toi',
        'pp_gf', 'pp_gpg',
        'pp_net_g', 'pp_net_gpg',
        'pp_pct', 'pp_net_pct',
        'sh_ga', 'sh_ga_pg',

        # PK stats
        'times_sh', 'times_sh_pg',
        'pk_net_pct', 'pk_pct',
        'pk_net_g', 'pk_net_gpg', 'pk_toi_pg',
        'pp_ga', 'pp_ga_pg',
        'sh_gf', 'sh_gf_pg',

        # cumulative stats
        'c_pp_opps', 'c_pp_net_g', 'net_pp_pct', 'ppng_rank',
        'c_pp_ga', 'c_times_sh', 'net_pk_pct', 'pk_rank',
    ]

    ordered_cols = [c for c in col_order if c in special_teams.columns]
    remaining_cols = [c for c in special_teams.columns if c not in ordered_cols]

    special_teams = special_teams[ordered_cols + remaining_cols]

    return special_teams


In [166]:
current_pk.columns

Index(['season_id', 'game_id', 'team_id', 'game_date', 'games_played',
       'home_road', 'losses', 'opponent', 'ot_losses', 'pk_net_pct', 'pk_pct',
       'pk_net_g', 'pk_net_gpg', 'pk_toi_pg', 'points', 'points_pct', 'pp_ga',
       'pp_ga_pg', 'sh_gf', 'sh_gf_pg', 'team_name', 'ties', 'times_sh',
       'times_sh_pg', 'wins', 'tri_code'],
      dtype='object')

In [167]:
hist_special_teams = build_special_teams_stats(history_pp, history_pk)
curr_special_teams = build_special_teams_stats(current_pp, current_pk)

curr_special_teams.head()

,season_id,game_id,team_id,tri_code,pp_opp,pp_opp_pg,pp_toi,pp_gf,pp_gpg,pp_net_g,pp_net_gpg,pp_pct,pp_net_pct,sh_ga,sh_ga_pg,times_sh,times_sh_pg,pk_net_pct,pk_pct,pk_net_g,pk_net_gpg,pk_toi_pg,pp_ga,pp_ga_pg,sh_gf,sh_gf_pg,c_pp_opps,c_pp_net_g,net_pp_pct,ppng_rank,c_pp_ga,c_times_sh,net_pk_pct,pk_rank,game_date
0,20252026,2025020014,1,NJD,2,2.0,151.0,0,0.0,0,0.0,0.00000,0.000000,0,0.0,4,4.0,0.750000,0.75,-1,-1.0,349.0,1,1.0,0,0.0,2.0,0.0,0.0,9,1.0,4.0,25.0,4,2025-10-09
1,20252026,2025020026,1,NJD,4,4.0,480.0,0,0.0,0,0.0,0.00000,0.000000,0,0.0,3,3.0,1.333333,1.00,1,1.0,349.0,0,0.0,1,1.0,6.0,0.0,0.0,16,1.0,7.0,14.3,5,2025-10-11
2,20252026,2025020045,1,NJD,2,2.0,205.0,2,2.0,2,2.0,1.00000,1.000000,0,0.0,5,5.0,1.000000,1.00,0,0.0,569.0,0,0.0,0,0.0,8.0,2.0,25.0,5,1.0,12.0,8.3,4,2025-10-13
3,20252026,2025020064,1,NJD,5,5.0,399.0,1,1.0,1,1.0,0.20000,0.200000,0,0.0,5,5.0,1.000000,1.00,0,0.0,518.0,0,0.0,0,0.0,13.0,3.0,23.1,5,1.0,17.0,5.9,2,2025-10-16
4,20252026,2025020078,1,NJD,3,3.0,244.0,1,1.0,1,1.0,0.33333,0.333333,0,0.0,3,3.0,1.333333,1.00,1,1.0,360.0,0,0.0,1,1.0,16.0,4.0,25.0,6,1.0,20.0,5.0,3,2025-10-18


## Final DFs

In [168]:
history_games = history_pbp[['season_id','game_id','team_id']].drop_duplicates()
history_games=history_games[history_games['team_id'].notna()].reset_index(drop=True)
history_games.head()

,season_id,game_id,team_id
0,20132014,2013020001,10
1,20132014,2013020001,8
2,20132014,2013020002,15
3,20132014,2013020002,16
4,20132014,2013020003,22


In [169]:
hist_cumulative_scoring_chances.columns

Index(['season_id', 'game_id', 'team_id', 'SCF', 'SCSF', 'SCGF', 'SCA', 'HDCF',
       'MDCF', 'LDCF', 'HDCA', 'MDCA', 'LDCA', 'HDCF_pct', 'HDCA_pct',
       'MDCF_pct', 'MDCA_pct', 'LDCF_pct', 'LDCA_pct', 'c_SCF', 'c_SCSF',
       'c_SCGF', 'c_SCA', 'SCF_pg', 'SCA_pg', 'c_HDCF', 'c_HDCA', 'c_MDCF',
       'c_MDCA', 'c_LDCF', 'c_LDCA', 'c_corsi_for', 'c_corsi_against',
       'c_corsi_diff', 'corsi_diff_per_game'],
      dtype='object')

In [173]:
def coalesce_duplicates(df, left="_x", right="_y"):
    for col in df.columns:
        if col.endswith(left):
            base = col[:-len(left)]
            other = base + right
            if other in df.columns:
                df[base] = df[col].combine_first(df[other])
                df.drop([col, other], axis=1, inplace=True)
    return df


history_df = (
    history_games
    .merge(history_game_summary[['game_id', 'game_date']], on='game_id', how='inner')
    .merge(hist_games_sos, on=['season_id', 'game_id', 'team_id'], how='inner')
    .merge(hist_special_teams, on=['season_id', 'game_id', 'team_id','tri_code'], how='inner')
    .merge(
        hist_cumulative_goalie_stats[['game_id', 'team_id', 'game_savePct', 'game_GAA']],
        on=['game_id', 'team_id'],
        how='inner'
    )
    .merge(history_cumulative_rush, on=['season_id','game_id', 'team_id'], how='inner')
    .merge(hist_cumulative_shots, on=['season_id','game_id', 'team_id'], how = 'inner' )
    .merge(hist_cumulative_scoring_chances, on=['season_id', 'game_id', 'team_id'], how='left')
    .merge(hist_cumulative_sog, on=['season_id', 'game_id', 'team_id',], how='inner' )

)
history_df = coalesce_duplicates(history_df)
history_df.head()

,season_id,game_id,team_id,tri_code,win,wins_before,games_before,win_pct_before,sos_past,sos_future,pp_opp,pp_opp_pg,pp_toi,pp_gf,pp_gpg,pp_net_g,pp_net_gpg,pp_pct,pp_net_pct,sh_ga,sh_ga_pg,times_sh,times_sh_pg,pk_net_pct,pk_pct,pk_net_g,pk_net_gpg,pk_toi_pg,pp_ga,pp_ga_pg,sh_gf,sh_gf_pg,c_pp_opps,c_pp_net_g,net_pp_pct,ppng_rank,c_pp_ga,c_times_sh,net_pk_pct,pk_rank,game_savePct,game_GAA,rush_shots_for,rush_shots_against,rush_shot_differential,c_rush_shots_for,c_rush_shots_against,c_rush_diff,rush_for_per_game,rush_against_per_game,rush_diff_per_game,shot_attempts_for,shot_attempts_against,shot_attempts_differential,corsi_for,corsi_against,corsi_diff,c_shots_diff,games_played,shot_attempts_for_per_game,shot_attempts_against_per_game,shots_diff_per_game,SCF,SCSF,SCGF,SCA,HDCF,MDCF,LDCF,HDCA,MDCA,LDCA,HDCF_pct,HDCA_pct,MDCF_pct,MDCA_pct,LDCF_pct,LDCA_pct,c_SCF,c_SCSF,c_SCGF,c_SCA,SCF_pg,SCA_pg,c_HDCF,c_HDCA,c_MDCF,c_MDCA,c_LDCF,c_LDCA,shots_for,goals_for,hd_sog,md_sog,ld_sog,hd_goals,md_goals,ld_goals,shot_pct,shots_against,goals_against,hd_against,md_against,ld_against,sv_pct,pdo,c_goals_for,c_hd_sog,c_md_sog,c_ld_sog,c_hd_goals,c_md_goals,c_ld_goals,c_goals_against,c_hd_against,c_md_against,c_ld_against,c_shot_pct,c_sv_pct,c_pdo,game_date,c_shots_for,c_shots_against,c_corsi_for,c_corsi_against,c_corsi_diff,corsi_diff_per_game
0,20132014,2013020001,10,TOR,1,<NA>,0,NaN,NaN,0.496280,7,7.0,700.0,1,1.0,1,1.0,0.14285,0.142857,0,0.0,4,4.0,1.250000,1.00000,1,1.0,405.0,0,0.0,1,1.0,7.0,1.0,14.3,4,0.0,4.0,0.0,1,0.91891,3.01003,9,7,2,9,7,2,9.0,7.0,2.0,61,61,0,45.0,44.0,1.0,0,1,61.0,61.0,0.0,31,38,4,65.0,7,27,28,6,25,32,0.538462,0.461538,0.519231,0.480769,0.466667,0.533333,31,38,4,65.0,31.0,65.0,7,6,27,25,28,32,38,4,5,15,18,1,2,1,10.526316,37,3,11,9,18,91.891892,102.418208,4,5,15,18,1,2,1,3,11,9,18,0.105263,0.918919,102.418208,2013-10-01,61,61,45.0,44.0,1.0,1.0
1,20132014,2013020001,8,MTL,0,<NA>,0,NaN,NaN,0.486969,4,4.0,405.0,0,0.0,-1,-1.0,0.00000,-0.250000,1,1.0,7,7.0,0.857142,0.85714,-1,-1.0,700.0,1,1.0,0,0.0,4.0,-1.0,-25.0,6,1.0,7.0,14.3,2,0.89473,4.06549,7,9,-2,7,9,-2,7.0,9.0,-2.0,70,69,1,53.0,53.0,0.0,1,1,70.0,69.0,1.0,34,37,3,65.0,15,20,38,16,22,34,0.483871,0.516129,0.476190,0.523810,0.527778,0.472222,34,37,3,65.0,34.0,65.0,15,16,20,22,38,34,37,3,11,9,18,3,0,0,8.108108,38,4,5,15,18,89.473684,97.581792,3,11,9,18,3,0,0,4,5,15,18,0.081081,0.894737,97.581792,2013-10-01,70,69,53.0,53.0,0.0,0.0
2,20132014,2013020002,15,WSH,0,<NA>,0,NaN,NaN,0.480114,6,6.0,388.0,3,3.0,3,3.0,0.50000,0.500000,0,0.0,4,4.0,0.750000,0.75000,-1,-1.0,387.0,1,1.0,0,0.0,6.0,3.0,50.0,1,1.0,4.0,25.0,3,0.85294,5.06329,4,7,-3,4,7,-3,4.0,7.0,-3.0,57,57,0,39.0,39.0,0.0,0,1,57.0,57.0,0.0,30,32,4,64.0,13,25,22,11,21,24,0.541667,0.458333,0.543478,0.456522,0.478261,0.521739,30,32,4,64.0,30.0,64.0,13,11,25,21,22,24,32,4,6,12,15,2,1,1,12.500000,35,6,7,9,20,82.857143,95.357143,4,6,12,15,2,1,1,6,7,9,20,0.125000,0.828571,95.357143,2013-10-01,57,57,39.0,39.0,0.0,0.0
3,20132014,2013020002,16,CHI,1,<NA>,0,NaN,NaN,0.507960,4,4.0,387.0,1,1.0,1,1.0,0.25000,0.250000,0,0.0,6,6.0,0.500000,0.50000,-3,-3.0,388.0,3,3.0,0,0.0,4.0,1.0,25.0,3,3.0,6.0,50.0,5,0.87500,4.00222,7,4,3,7,4,3,7.0,4.0,3.0,55,54,1,38.0,38.0,0.0,1,1,55.0,54.0,1.0,34,35,6,64.0,13,18,26,15,22,24,0.464286,0.535714,0.450000,0.550000,0.520000,0.480000,34,35,6,64.0,34.0,64.0,13,15,18,22,26,24,35,6,7,9,20,3,2,1,17.142857,32,4,6,12,15,87.500000,104.642857,6,7,9,20,3,2,1,4,6,12,15,0.171429,0.875000,104.642857,2013-10-01,55,54,38.0,38.0,0.0,0.0
4,20132014,2013020003,22,EDM,0,<NA>,0,NaN,NaN,0.507925,5,5.0,399.0,2,2.0,2,2.0,0.40000,0.400000,0,0.0,4,4.0,0.750000,0.75000,-1,-1.0,400.0,1,1.0,0,0.0,5.0,2.0,40.0,2,1.0,4.0,25.0,3,0.82142,5.18283,9,4,5,9,4,5,9.0,4.0,5.0,71,70,1,48.0,49.0,-1.0,1,1,71.0,70.0,1.0,38,38,4,76.0,15,29,28,11,28,24,0.576923,0.423077,0.508772,0.491228,0.538462,0.461538,38,38,4,76.0,38.0,76.0,15,11,29,28,28,24,38,4,9,15,14,2,2,0,10.526316,28,5,7,12,10,82.142857,92.669173,4,9,15,14,2,2,0,5,7,12,10,0.105263,0.821429,92.6

In [174]:
def coalesce_dupes(df, left="_x", right="_y"):
    for col in df.columns:
        if col.endswith(left):
            base = col[:-len(left)]
            other = base + right
            if other in df.columns:
                df[base] = df[col].combine_first(df[other])
                df.drop([col, other], axis=1, inplace=True)
    return df
history_df = history_df.merge(history_daily_standings, on=[ 'season_id', 'team_id','tri_code','game_date'], how = 'inner')
history_df = coalesce_dupes(history_df)
history_df.head()

,season_id,game_id,team_id,tri_code,win,wins_before,games_before,win_pct_before,sos_past,sos_future,pp_opp,pp_opp_pg,pp_toi,pp_gf,pp_gpg,pp_net_g,pp_net_gpg,pp_pct,pp_net_pct,sh_ga,sh_ga_pg,times_sh,times_sh_pg,pk_net_pct,pk_pct,pk_net_g,pk_net_gpg,pk_toi_pg,pp_ga,pp_ga_pg,sh_gf,sh_gf_pg,c_pp_opps,c_pp_net_g,net_pp_pct,ppng_rank,c_pp_ga,c_times_sh,net_pk_pct,pk_rank,game_savePct,game_GAA,rush_shots_for,rush_shots_against,rush_shot_differential,c_rush_shots_for,c_rush_shots_against,c_rush_diff,rush_for_per_game,rush_against_per_game,rush_diff_per_game,shot_attempts_for,shot_attempts_against,shot_attempts_differential,corsi_for,corsi_against,corsi_diff,c_shots_diff,games_played,shot_attempts_for_per_game,shot_attempts_against_per_game,shots_diff_per_game,SCF,SCSF,SCGF,SCA,HDCF,MDCF,LDCF,HDCA,MDCA,LDCA,HDCF_pct,HDCA_pct,MDCF_pct,MDCA_pct,LDCF_pct,LDCA_pct,c_SCF,c_SCSF,c_SCGF,c_SCA,SCF_pg,SCA_pg,c_HDCF,c_HDCA,c_MDCF,c_MDCA,c_LDCF,c_LDCA,shots_for,hd_sog,md_sog,ld_sog,hd_goals,md_goals,ld_goals,shot_pct,shots_against,hd_against,md_against,ld_against,sv_pct,pdo,c_goals_for,c_hd_sog,c_md_sog,c_ld_sog,c_hd_goals,c_md_goals,c_ld_goals,c_goals_against,c_hd_against,c_md_against,c_ld_against,c_shot_pct,c_sv_pct,c_pdo,game_date,c_shots_for,c_shots_against,c_corsi_for,c_corsi_against,c_corsi_diff,corsi_diff_per_game,team_name,games_remaining,wins,wins_pctg,losses,ot_losses,rotw,rotw_pctg,rw,so_losses,so_wins,points,points_pct,projected_points,in_playoff_spot,gf_pctg,goal_diff,goal_diff_pctg,streak,streak_count,last_10_gp,last_10_wins,last_10_losses,last_10_otl,last_10_points,last_10_gf,last_10_ga,last_10_goal_diff,last_10_rotw,last_10_rw,home_gp,home_wins,home_losses,home_otlosses,home_rotw,home_rw,home_points,home_gf,home_ga,home_goal_diff,road_gp,road_wins,road_losses,road_ot_losses,road_rotw,road_rw,road_points,road_gf,road_ga,road_goal_diff,division,division_rank,division_home_rank,division_road_rank,division_last_10_rank,conference,conference_rank,conference_home_rank,conference_road_rank,conference_last_10_rank,league_rank,league_home_rank,league_road_rank,league_last_10_rank,wildcard_rank,made_playoffs,goals_for,goals_against
0,20132014,2013020001,10,TOR,1,<NA>,0,NaN,NaN,0.496280,7,7.0,700.0,1,1.0,1,1.0,0.14285,0.142857,0,0.0,4,4.0,1.250000,1.00000,1,1.0,405.0,0,0.0,1,1.0,7.0,1.0,14.3,4,0.0,4.0,0.0,1,0.91891,3.01003,9,7,2,9,7,2,9.0,7.0,2.0,61,61,0,45.0,44.0,1.0,0,1,61.0,61.0,0.0,31,38,4,65.0,7,27,28,6,25,32,0.538462,0.461538,0.519231,0.480769,0.466667,0.533333,31,38,4,65.0,31.0,65.0,7,6,27,25,28,32,38,5,15,18,1,2,1,10.526316,37,11,9,18,91.891892,102.418208,4,5,15,18,1,2,1,3,11,9,18,0.105263,0.918919,102.418208,2013-10-01,61,61,45.0,44.0,1.0,1.0,Toronto Maple Leafs,81,1,1.0,0,0,1,1.0,1,0,0,2,1.0,164,0,4.0,1,1.0,W,1.0,1,1,0,0,2,4,3,1,1,1,0,0,0,0,0,0,0,0,0,0,1,1,0,0,1,1,2,4,3,1,Atlantic,1,7,1,1,Eastern,1,14,1,1,2,25,1,2,0,0,4,3
1,20132014,2013020001,8,MTL,0,<NA>,0,NaN,NaN,0.486969,4,4.0,405.0,0,0.0,-1,-1.0,0.00000,-0.250000,1,1.0,7,7.0,0.857142,0.85714,-1,-1.0,700.0,1,1.0,0,0.0,4.0,-1.0,-25.0,6,1.0,7.0,14.3,2,0.89473,4.06549,7,9,-2,7,9,-2,7.0,9.0,-2.0,70,69,1,53.0,53.0,0.0,1,1,70.0,69.0,1.0,34,37,3,65.0,15,20,38,16,22,34,0.483871,0.516129,0.476190,0.523810,0.527778,0.472222,34,37,3,65.0,34.0,65.0,15,16,20,22,38,34,37,11,9,18,3,0,0,8.108108,38,5,15,18,89.473684,97.581792,3,11,9,18,3,0,0,4,5,15,18,0.081081,0.894737,97.581792,2013-10-01,70,69,53.0,53.0,0.0,0.0,Montréal Canadiens,81,0,0.0,1,0,0,0.0,0,0,0,0,0.0,0,0,3.0,-1,-1.0,L,1.0,1,0,1,0,0,3,4,-1,0,0,1,0,1,0,0,0,0,3,4,-1,0,0,0,0,0,0,0,0,0,0,Atlantic,8,8,6,8,Eastern,15,16,8,15,29,30,17,29,9,1,3,4
2,20132014,2013020002,15,WSH,0,<NA>,0,NaN,NaN,0.480114,6,6.0,388.0,3,3.0,3,3.0,0.50000,0.500000,0,0.0,4,4.0,0.750000,0.75000,-1,-1.0,387.0,1,1.0,0,0.0,6.0,3.0,50.0,1,1.0,4.0,25.0,3,0.85294,5.06329,4,7,-3,4,7,-3,4.0,7.0,-3.0,57,57,0,39.0,39.0,0.0,0,1,57.0,57.0,0.0,30,32,4,64.0,13,25,22,11,21,24,0.541667,0.458333,0.543478,0.456522,0.478261,0.521739,30,32,4,64.0,30.0,64.0,13,11,25,21,22,24,32,6,12,15

In [175]:
cols = list(history_df.columns)

# remove team_id from its current position
cols.remove("game_date")
cols.remove("team_name")


# insert it at position 2 (3rd column, excluding index)
cols.insert(2, "game_date")
cols.insert(5, "team_name")

# reassign columns
history_df = history_df[cols]
history_df.head(2)

,season_id,game_id,game_date,team_id,tri_code,team_name,win,wins_before,games_before,win_pct_before,sos_past,sos_future,pp_opp,pp_opp_pg,pp_toi,pp_gf,pp_gpg,pp_net_g,pp_net_gpg,pp_pct,pp_net_pct,sh_ga,sh_ga_pg,times_sh,times_sh_pg,pk_net_pct,pk_pct,pk_net_g,pk_net_gpg,pk_toi_pg,pp_ga,pp_ga_pg,sh_gf,sh_gf_pg,c_pp_opps,c_pp_net_g,net_pp_pct,ppng_rank,c_pp_ga,c_times_sh,net_pk_pct,pk_rank,game_savePct,game_GAA,rush_shots_for,rush_shots_against,rush_shot_differential,c_rush_shots_for,c_rush_shots_against,c_rush_diff,rush_for_per_game,rush_against_per_game,rush_diff_per_game,shot_attempts_for,shot_attempts_against,shot_attempts_differential,corsi_for,corsi_against,corsi_diff,c_shots_diff,games_played,shot_attempts_for_per_game,shot_attempts_against_per_game,shots_diff_per_game,SCF,SCSF,SCGF,SCA,HDCF,MDCF,LDCF,HDCA,MDCA,LDCA,HDCF_pct,HDCA_pct,MDCF_pct,MDCA_pct,LDCF_pct,LDCA_pct,c_SCF,c_SCSF,c_SCGF,c_SCA,SCF_pg,SCA_pg,c_HDCF,c_HDCA,c_MDCF,c_MDCA,c_LDCF,c_LDCA,shots_for,hd_sog,md_sog,ld_sog,hd_goals,md_goals,ld_goals,shot_pct,shots_against,hd_against,md_against,ld_against,sv_pct,pdo,c_goals_for,c_hd_sog,c_md_sog,c_ld_sog,c_hd_goals,c_md_goals,c_ld_goals,c_goals_against,c_hd_against,c_md_against,c_ld_against,c_shot_pct,c_sv_pct,c_pdo,c_shots_for,c_shots_against,c_corsi_for,c_corsi_against,c_corsi_diff,corsi_diff_per_game,games_remaining,wins,wins_pctg,losses,ot_losses,rotw,rotw_pctg,rw,so_losses,so_wins,points,points_pct,projected_points,in_playoff_spot,gf_pctg,goal_diff,goal_diff_pctg,streak,streak_count,last_10_gp,last_10_wins,last_10_losses,last_10_otl,last_10_points,last_10_gf,last_10_ga,last_10_goal_diff,last_10_rotw,last_10_rw,home_gp,home_wins,home_losses,home_otlosses,home_rotw,home_rw,home_points,home_gf,home_ga,home_goal_diff,road_gp,road_wins,road_losses,road_ot_losses,road_rotw,road_rw,road_points,road_gf,road_ga,road_goal_diff,division,division_rank,division_home_rank,division_road_rank,division_last_10_rank,conference,conference_rank,conference_home_rank,conference_road_rank,conference_last_10_rank,league_rank,league_home_rank,league_road_rank,league_last_10_rank,wildcard_rank,made_playoffs,goals_for,goals_against
0,20132014,2013020001,2013-10-01,10,TOR,Toronto Maple Leafs,1,<NA>,0,NaN,NaN,0.496280,7,7.0,700.0,1,1.0,1,1.0,0.14285,0.142857,0,0.0,4,4.0,1.250000,1.00000,1,1.0,405.0,0,0.0,1,1.0,7.0,1.0,14.3,4,0.0,4.0,0.0,1,0.91891,3.01003,9,7,2,9,7,2,9.0,7.0,2.0,61,61,0,45.0,44.0,1.0,0,1,61.0,61.0,0.0,31,38,4,65.0,7,27,28,6,25,32,0.538462,0.461538,0.519231,0.480769,0.466667,0.533333,31,38,4,65.0,31.0,65.0,7,6,27,25,28,32,38,5,15,18,1,2,1,10.526316,37,11,9,18,91.891892,102.418208,4,5,15,18,1,2,1,3,11,9,18,0.105263,0.918919,102.418208,61,61,45.0,44.0,1.0,1.0,81,1,1.0,0,0,1,1.0,1,0,0,2,1.0,164,0,4.0,1,1.0,W,1.0,1,1,0,0,2,4,3,1,1,1,0,0,0,0,0,0,0,0,0,0,1,1,0,0,1,1,2,4,3,1,Atlantic,1,7,1,1,Eastern,1,14,1,1,2,25,1,2,0,0,4,3
1,20132014,2013020001,2013-10-01,8,MTL,Montréal Canadiens,0,<NA>,0,NaN,NaN,0.486969,4,4.0,405.0,0,0.0,-1,-1.0,0.00000,-0.250000,1,1.0,7,7.0,0.857142,0.85714,-1,-1.0,700.0,1,1.0,0,0.0,4.0,-1.0,-25.0,6,1.0,7.0,14.3,2,0.89473,4.06549,7,9,-2,7,9,-2,7.0,9.0,-2.0,70,69,1,53.0,53.0,0.0,1,1,70.0,69.0,1.0,34,37,3,65.0,15,20,38,16,22,34,0.483871,0.516129,0.476190,0.523810,0.527778,0.472222,34,37,3,65.0,34.0,65.0,15,16,20,22,38,34,37,11,9,18,3,0,0,8.108108,38,5,15,18,89.473684,97.581792,3,11,9,18,3,0,0,4,5,15,18,0.081081,0.894737,97.581792,70,69,53.0,53.0,0.0,0.0,81,0,0.0,1,0,0,0.0,0,0,0,0,0.0,0,0,3.0,-1,-1.0,L,1.0,1,0,1,0,0,3,4,-1,0,0,1,0,1,0,0,0,0,3,4,-1,0,0,0,0,0,0,0,0,0,0,Atlantic,8,8,6,8,Eastern,15,16,8,15,29,30,17,29,9,1,3,4


In [176]:
current_games = current_pbp[['season_id','game_id','team_id']].drop_duplicates()
current_games=current_games[current_games['team_id'].notna()].reset_index(drop=True)
current_games.head()

,season_id,game_id,team_id
0,20252026,2025020001,16
1,20252026,2025020001,13
2,20252026,2025020002,3
3,20252026,2025020002,5
4,20252026,2025020003,21


In [177]:
def coalesce_duplicates(df, left="_x", right="_y"):
    for col in df.columns:
        if col.endswith(left):
            base = col[:-len(left)]
            other = base + right
            if other in df.columns:
                df[base] = df[col].combine_first(df[other])
                df.drop([col, other], axis=1, inplace=True)
    return df

curr_df = (
    current_games
    .merge(game_summary[['game_id', 'game_date']], on='game_id', how='inner')
    .merge(curr_games_sos, on=['season_id', 'game_id', 'team_id'], how='inner')
    .merge(curr_special_teams, on=['season_id', 'game_id', 'team_id','tri_code'], how='inner')
    .merge(
        curr_cumulative_goalie_stats[['game_id', 'team_id', 'game_savePct', 'game_GAA']],
        on=['game_id', 'team_id'],
        how='inner'
    )
    .merge(current_cumulative_rush, on=['season_id','game_id', 'team_id'], how='inner')
    .merge(curr_cumulative_shots, on=['season_id','game_id', 'team_id'], how = 'inner' )
    .merge(curr_cumulative_scoring_chances, on=['season_id', 'game_id', 'team_id'], how='left')
    .merge(curr_cumulative_sog, on=['season_id', 'game_id', 'team_id',], how='inner' )
)

curr_df = coalesce_duplicates(curr_df)
curr_df.head()

,season_id,game_id,team_id,tri_code,win,wins_before,games_before,win_pct_before,sos_past,sos_future,pp_opp,pp_opp_pg,pp_toi,pp_gf,pp_gpg,pp_net_g,pp_net_gpg,pp_pct,pp_net_pct,sh_ga,sh_ga_pg,times_sh,times_sh_pg,pk_net_pct,pk_pct,pk_net_g,pk_net_gpg,pk_toi_pg,pp_ga,pp_ga_pg,sh_gf,sh_gf_pg,c_pp_opps,c_pp_net_g,net_pp_pct,ppng_rank,c_pp_ga,c_times_sh,net_pk_pct,pk_rank,game_savePct,game_GAA,rush_shots_for,rush_shots_against,rush_shot_differential,c_rush_shots_for,c_rush_shots_against,c_rush_diff,rush_for_per_game,rush_against_per_game,rush_diff_per_game,shot_attempts_for,shot_attempts_against,shot_attempts_differential,corsi_for,corsi_against,corsi_diff,c_shots_diff,games_played,shot_attempts_for_per_game,shot_attempts_against_per_game,shots_diff_per_game,SCF,SCSF,SCGF,SCA,HDCF,MDCF,LDCF,HDCA,MDCA,LDCA,HDCF_pct,HDCA_pct,MDCF_pct,MDCA_pct,LDCF_pct,LDCA_pct,c_SCF,c_SCSF,c_SCGF,c_SCA,SCF_pg,SCA_pg,c_HDCF,c_HDCA,c_MDCF,c_MDCA,c_LDCF,c_LDCA,shots_for,goals_for,hd_sog,md_sog,ld_sog,hd_goals,md_goals,ld_goals,shot_pct,shots_against,goals_against,hd_against,md_against,ld_against,sv_pct,pdo,c_goals_for,c_hd_sog,c_md_sog,c_ld_sog,c_hd_goals,c_md_goals,c_ld_goals,c_goals_against,c_hd_against,c_md_against,c_ld_against,c_shot_pct,c_sv_pct,c_pdo,game_date,c_shots_for,c_shots_against,c_corsi_for,c_corsi_against,c_corsi_diff,corsi_diff_per_game
0,20252026,2025020001,16,CHI,0,<NA>,0,NaN,NaN,0.497453,3,3.0,360.0,0,0.0,0,0.0,0.00000,0.000000,0,0.0,2,2.0,0.50,0.50,-1,-1.0,222.0,1,1.0,0,0.0,3.0,0.0,0.0,4,1.0,2.0,50.0,4,0.91891,3.04740,2,8,-6,2,8,-6,2.0,8.0,-6.0,50,49,1,33,34,-1,1,1,50.0,49.0,1.0,28,19,2,65,11,14,26,8,21,23,0.578947,0.421053,0.400000,0.600000,0.530612,0.469388,28,19,2,65,28.0,65.0,11,8,14,21,26,23,19,2,3,5,12,0,2,0,10.526316,37,3,10,11,16,91.891892,102.418208,2,3,5,12,0,2,0,3,10,11,16,0.105263,0.918919,102.418208,2025-10-07,50,49,33,34,-1,-1.0
1,20252026,2025020001,13,FLA,1,<NA>,0,NaN,NaN,0.497126,2,2.0,222.0,1,1.0,1,1.0,0.50000,0.500000,0,0.0,3,3.0,1.00,1.00,0,0.0,360.0,0,0.0,0,0.0,2.0,1.0,50.0,1,0.0,3.0,0.0,1,0.89473,2.00000,12,6,6,12,6,6,12.0,6.0,6.0,69,69,0,53,52,1,0,1,69.0,69.0,0.0,37,37,3,65,17,24,28,20,17,31,0.459459,0.540541,0.585366,0.414634,0.474576,0.525424,37,37,3,65,37.0,65.0,17,20,24,17,28,31,37,3,10,11,16,2,0,1,8.108108,19,2,3,5,12,89.473684,97.581792,3,10,11,16,2,0,1,2,3,5,12,0.081081,0.894737,97.581792,2025-10-07,69,69,53,52,1,1.0
2,20252026,2025020002,3,NYR,0,<NA>,0,NaN,NaN,0.513884,2,2.0,240.0,0,0.0,0,0.0,0.00000,0.000000,0,0.0,1,1.0,1.00,1.00,0,0.0,120.0,0,0.0,0,0.0,2.0,0.0,0.0,4,0.0,1.0,0.0,1,0.96551,1.00642,7,9,-2,7,9,-2,7.0,9.0,-2.0,60,60,0,36,36,0,0,1,60.0,60.0,0.0,34,25,0,66,14,19,29,16,19,24,0.466667,0.533333,0.500000,0.500000,0.547170,0.452830,34,25,0,66,34.0,66.0,14,16,19,19,29,24,25,0,4,8,14,0,0,0,0.000000,31,3,10,7,14,90.322581,90.322581,0,4,8,14,0,0,0,3,10,7,14,0.000000,0.903226,90.322581,2025-10-07,60,60,36,36,0,0.0
3,20252026,2025020002,5,PIT,1,<NA>,0,NaN,NaN,0.469285,1,1.0,120.0,0,0.0,0,0.0,0.00000,0.000000,0,0.0,2,2.0,1.00,1.00,0,0.0,240.0,0,0.0,0,0.0,1.0,0.0,0.0,4,0.0,2.0,0.0,1,1.00000,0.00000,9,7,2,9,7,2,9.0,7.0,2.0,61,60,1,40,39,1,1,1,61.0,60.0,1.0,32,31,3,66,18,17,26,16,17,31,0.529412,0.470588,0.500000,0.500000,0.456140,0.543860,32,31,3,66,32.0,66.0,18,16,17,17,26,31,31,3,10,7,14,2,0,1,9.677419,25,0,4,8,14,100.000000,109.677419,3,10,7,14,2,0,1,0,4,8,14,0.096774,1.000000,109.677419,2025-10-07,61,60,40,39,1,1.0
4,20252026,2025020003,21,COL,1,<NA>,0,NaN,NaN,0.521309,6,6.0,605.0,1,1.0,1,1.0,0.16666,0.166666,0,0.0,4,4.0,0.75,0.75,-1,-1.0,438.0,1,1.0,0,0.0,6.0,1.0,16.7,3,1.0,4.0,25.0,3,0.96000,1.00194,7,6,1,7,6,1,7.0,6.0,1.0,49,48,1,35,29,6,1,1,49.0,48.0,1.0,27,23,4,52,13,12,26,17,12,30,0.433333,0.566667,0.500000,0.500000,0.464286,0.535714,27,23,4,52,27.0,52.0,13,17,12,12,26,30,23,4,7,3,14,1,1,2,17.391304,25,1,7,6,14,96.000000,113.391304,4,7,3,14,1,1,2,1,7,6,14,0.173913,0.960000,113.391304,2025-10-07,49,48,35,29,6,6.0


In [178]:
def coalesce_dupes(df, left="_x", right="_y"):
    for col in df.columns:
        if col.endswith(left):
            base = col[:-len(left)]
            other = base + right
            if other in df.columns:
                df[base] = df[col].combine_first(df[other])
                df.drop([col, other], axis=1, inplace=True)
    return df

curr_df = curr_df.merge(daily_standings, on=[ 'season_id', 'team_id','tri_code','game_date'], how = 'inner')

curr_df = coalesce_dupes(curr_df)
curr_df.head()

,season_id,game_id,team_id,tri_code,win,wins_before,games_before,win_pct_before,sos_past,sos_future,pp_opp,pp_opp_pg,pp_toi,pp_gf,pp_gpg,pp_net_g,pp_net_gpg,pp_pct,pp_net_pct,sh_ga,sh_ga_pg,times_sh,times_sh_pg,pk_net_pct,pk_pct,pk_net_g,pk_net_gpg,pk_toi_pg,pp_ga,pp_ga_pg,sh_gf,sh_gf_pg,c_pp_opps,c_pp_net_g,net_pp_pct,ppng_rank,c_pp_ga,c_times_sh,net_pk_pct,pk_rank,game_savePct,game_GAA,rush_shots_for,rush_shots_against,rush_shot_differential,c_rush_shots_for,c_rush_shots_against,c_rush_diff,rush_for_per_game,rush_against_per_game,rush_diff_per_game,shot_attempts_for,shot_attempts_against,shot_attempts_differential,corsi_for,corsi_against,corsi_diff,c_shots_diff,games_played,shot_attempts_for_per_game,shot_attempts_against_per_game,shots_diff_per_game,SCF,SCSF,SCGF,SCA,HDCF,MDCF,LDCF,HDCA,MDCA,LDCA,HDCF_pct,HDCA_pct,MDCF_pct,MDCA_pct,LDCF_pct,LDCA_pct,c_SCF,c_SCSF,c_SCGF,c_SCA,SCF_pg,SCA_pg,c_HDCF,c_HDCA,c_MDCF,c_MDCA,c_LDCF,c_LDCA,shots_for,hd_sog,md_sog,ld_sog,hd_goals,md_goals,ld_goals,shot_pct,shots_against,hd_against,md_against,ld_against,sv_pct,pdo,c_goals_for,c_hd_sog,c_md_sog,c_ld_sog,c_hd_goals,c_md_goals,c_ld_goals,c_goals_against,c_hd_against,c_md_against,c_ld_against,c_shot_pct,c_sv_pct,c_pdo,game_date,c_shots_for,c_shots_against,c_corsi_for,c_corsi_against,c_corsi_diff,corsi_diff_per_game,team_name,games_remaining,wins,wins_pctg,losses,ot_losses,rotw,rotw_pctg,rw,so_losses,so_wins,points,points_pct,projected_points,in_playoff_spot,gf_pctg,goal_diff,goal_diff_pctg,streak,streak_count,last_10_gp,last_10_wins,last_10_losses,last_10_otl,last_10_points,last_10_gf,last_10_ga,last_10_goal_diff,last_10_rotw,last_10_rw,home_gp,home_wins,home_losses,home_otlosses,home_rotw,home_rw,home_points,home_gf,home_ga,home_goal_diff,road_gp,road_wins,road_losses,road_ot_losses,road_rotw,road_rw,road_points,road_gf,road_ga,road_goal_diff,division,division_rank,division_home_rank,division_road_rank,division_last_10_rank,conference,conference_rank,conference_home_rank,conference_road_rank,conference_last_10_rank,league_rank,league_home_rank,league_road_rank,league_last_10_rank,wildcard_rank,goals_for,goals_against
0,20252026,2025020001,16,CHI,0,<NA>,0,NaN,NaN,0.497453,3,3.0,360.0,0,0.0,0,0.0,0.00000,0.000000,0,0.0,2,2.0,0.50,0.50,-1,-1.0,222.0,1,1.0,0,0.0,3.0,0.0,0.0,4,1.0,2.0,50.0,4,0.91891,3.04740,2,8,-6,2,8,-6,2.0,8.0,-6.0,50,49,1,33,34,-1,1,1,50.0,49.0,1.0,28,19,2,65,11,14,26,8,21,23,0.578947,0.421053,0.400000,0.600000,0.530612,0.469388,28,19,2,65,28.0,65.0,11,8,14,21,26,23,19,3,5,12,0,2,0,10.526316,37,10,11,16,91.891892,102.418208,2,3,5,12,0,2,0,3,10,11,16,0.105263,0.918919,102.418208,2025-10-07,50,49,33,34,-1,-1.0,Chicago Blackhawks,81,0,0.0,1,0,0,0.0,0,0,0,0,0.0,0,0,2.0,-1,-1.0,L,1.0,1,0,1,0,0,2,3,-1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,2,3,-1,Central,8,1,8,8,Western,15,3,16,15,30,7,32,30,9,2,3
1,20252026,2025020001,13,FLA,1,<NA>,0,NaN,NaN,0.497126,2,2.0,222.0,1,1.0,1,1.0,0.50000,0.500000,0,0.0,3,3.0,1.00,1.00,0,0.0,360.0,0,0.0,0,0.0,2.0,1.0,50.0,1,0.0,3.0,0.0,1,0.89473,2.00000,12,6,6,12,6,6,12.0,6.0,6.0,69,69,0,53,52,1,0,1,69.0,69.0,0.0,37,37,3,65,17,24,28,20,17,31,0.459459,0.540541,0.585366,0.414634,0.474576,0.525424,37,37,3,65,37.0,65.0,17,20,24,17,28,31,37,10,11,16,2,0,1,8.108108,19,3,5,12,89.473684,97.581792,3,10,11,16,2,0,1,2,3,5,12,0.081081,0.894737,97.581792,2025-10-07,69,69,53,52,1,1.0,Florida Panthers,81,1,1.0,0,0,1,1.0,1,0,0,2,1.0,164,0,3.0,1,1.0,W,1.0,1,1,0,0,2,3,2,1,1,1,1,1,0,0,1,1,2,3,2,1,0,0,0,0,0,0,0,0,0,0,Atlantic,1,1,4,1,Eastern,2,1,7,2,3,1,12,3,0,3,2
2,20252026,2025020002,3,NYR,0,<NA>,0,NaN,NaN,0.513884,2,2.0,240.0,0,0.0,0,0.0,0.00000,0.000000,0,0.0,1,1.0,1.00,1.00,0,0.0,120.0,0,0.0,0,0.0,2.0,0.0,0.0,4,0.0,1.0,0.0,1,0.96551,1.00642,7,9,-2,7,9,-2,7.0,9.0,-2.0,60,60,0,36,36,0,0,1,60.0,60.0,0.0,34,25,0,66,14,19,29,16,19,24,0.466667,0.533333,0.500000,0.500000,0.547170,0.452830,34,25,0,66,34.0,66.0,14,16,19,19,29,24,25,4,8,14,0,0,0,0.000000,31,10,7,14,90.322581,90.322581,0,4,8,14,0,0,0,3,10,7,14,0.000000,0.903226,

In [179]:
cols = list(curr_df.columns)

# remove team_id from its current position
cols.remove("game_date")
cols.remove("team_name")


# insert it at position 2 (3rd column, excluding index)
cols.insert(2, "game_date")
cols.insert(5, "team_name")


# reassign columns
curr_df = curr_df[cols]
curr_df.head(2)

,season_id,game_id,game_date,team_id,tri_code,team_name,win,wins_before,games_before,win_pct_before,sos_past,sos_future,pp_opp,pp_opp_pg,pp_toi,pp_gf,pp_gpg,pp_net_g,pp_net_gpg,pp_pct,pp_net_pct,sh_ga,sh_ga_pg,times_sh,times_sh_pg,pk_net_pct,pk_pct,pk_net_g,pk_net_gpg,pk_toi_pg,pp_ga,pp_ga_pg,sh_gf,sh_gf_pg,c_pp_opps,c_pp_net_g,net_pp_pct,ppng_rank,c_pp_ga,c_times_sh,net_pk_pct,pk_rank,game_savePct,game_GAA,rush_shots_for,rush_shots_against,rush_shot_differential,c_rush_shots_for,c_rush_shots_against,c_rush_diff,rush_for_per_game,rush_against_per_game,rush_diff_per_game,shot_attempts_for,shot_attempts_against,shot_attempts_differential,corsi_for,corsi_against,corsi_diff,c_shots_diff,games_played,shot_attempts_for_per_game,shot_attempts_against_per_game,shots_diff_per_game,SCF,SCSF,SCGF,SCA,HDCF,MDCF,LDCF,HDCA,MDCA,LDCA,HDCF_pct,HDCA_pct,MDCF_pct,MDCA_pct,LDCF_pct,LDCA_pct,c_SCF,c_SCSF,c_SCGF,c_SCA,SCF_pg,SCA_pg,c_HDCF,c_HDCA,c_MDCF,c_MDCA,c_LDCF,c_LDCA,shots_for,hd_sog,md_sog,ld_sog,hd_goals,md_goals,ld_goals,shot_pct,shots_against,hd_against,md_against,ld_against,sv_pct,pdo,c_goals_for,c_hd_sog,c_md_sog,c_ld_sog,c_hd_goals,c_md_goals,c_ld_goals,c_goals_against,c_hd_against,c_md_against,c_ld_against,c_shot_pct,c_sv_pct,c_pdo,c_shots_for,c_shots_against,c_corsi_for,c_corsi_against,c_corsi_diff,corsi_diff_per_game,games_remaining,wins,wins_pctg,losses,ot_losses,rotw,rotw_pctg,rw,so_losses,so_wins,points,points_pct,projected_points,in_playoff_spot,gf_pctg,goal_diff,goal_diff_pctg,streak,streak_count,last_10_gp,last_10_wins,last_10_losses,last_10_otl,last_10_points,last_10_gf,last_10_ga,last_10_goal_diff,last_10_rotw,last_10_rw,home_gp,home_wins,home_losses,home_otlosses,home_rotw,home_rw,home_points,home_gf,home_ga,home_goal_diff,road_gp,road_wins,road_losses,road_ot_losses,road_rotw,road_rw,road_points,road_gf,road_ga,road_goal_diff,division,division_rank,division_home_rank,division_road_rank,division_last_10_rank,conference,conference_rank,conference_home_rank,conference_road_rank,conference_last_10_rank,league_rank,league_home_rank,league_road_rank,league_last_10_rank,wildcard_rank,goals_for,goals_against
0,20252026,2025020001,2025-10-07,16,CHI,Chicago Blackhawks,0,<NA>,0,NaN,NaN,0.497453,3,3.0,360.0,0,0.0,0,0.0,0.0,0.0,0,0.0,2,2.0,0.5,0.5,-1,-1.0,222.0,1,1.0,0,0.0,3.0,0.0,0.0,4,1.0,2.0,50.0,4,0.91891,3.0474,2,8,-6,2,8,-6,2.0,8.0,-6.0,50,49,1,33,34,-1,1,1,50.0,49.0,1.0,28,19,2,65,11,14,26,8,21,23,0.578947,0.421053,0.400000,0.600000,0.530612,0.469388,28,19,2,65,28.0,65.0,11,8,14,21,26,23,19,3,5,12,0,2,0,10.526316,37,10,11,16,91.891892,102.418208,2,3,5,12,0,2,0,3,10,11,16,0.105263,0.918919,102.418208,50,49,33,34,-1,-1.0,81,0,0.0,1,0,0,0.0,0,0,0,0,0.0,0,0,2.0,-1,-1.0,L,1.0,1,0,1,0,0,2,3,-1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,2,3,-1,Central,8,1,8,8,Western,15,3,16,15,30,7,32,30,9,2,3
1,20252026,2025020001,2025-10-07,13,FLA,Florida Panthers,1,<NA>,0,NaN,NaN,0.497126,2,2.0,222.0,1,1.0,1,1.0,0.5,0.5,0,0.0,3,3.0,1.0,1.0,0,0.0,360.0,0,0.0,0,0.0,2.0,1.0,50.0,1,0.0,3.0,0.0,1,0.89473,2.0000,12,6,6,12,6,6,12.0,6.0,6.0,69,69,0,53,52,1,0,1,69.0,69.0,0.0,37,37,3,65,17,24,28,20,17,31,0.459459,0.540541,0.585366,0.414634,0.474576,0.525424,37,37,3,65,37.0,65.0,17,20,24,17,28,31,37,10,11,16,2,0,1,8.108108,19,3,5,12,89.473684,97.581792,3,10,11,16,2,0,1,2,3,5,12,0.081081,0.894737,97.581792,69,69,53,52,1,1.0,81,1,1.0,0,0,1,1.0,1,0,0,2,1.0,164,0,3.0,1,1.0,W,1.0,1,1,0,0,2,3,2,1,1,1,1,1,0,0,1,1,2,3,2,1,0,0,0,0,0,0,0,0,0,0,Atlantic,1,1,4,1,Eastern,2,1,7,2,3,1,12,3,0,3,2


In [180]:
#special features

def special_features(df):
    df['ot_wins'] = (df['rotw']-df['rw']).clip(lower=0)
    df['overtimes'] = df['ot_losses']+df['ot_wins']
    df["ot_games_pct"] = (
        df["overtimes"] / df["games_played"]
    ).fillna(0).round(2)


    # Point% × Strength of Schedule
    df["pt_pct_x_sos"] = df["points_pct"] * df["sos_future"]

    # Goalie GA × SV% interaction
    df["gaa_x_sv"] = df["game_GAA"] * df["game_savePct"]

    # Goal differential × Rush differential
    df["gd_x_rushdiff"] = (
        df["goal_diff"] * df["rush_shot_differential"]
    )

    # PK workload × PK%
    df["pk_load_x_pk_pct"] = (
        df["times_sh_pg"] * df["pk_pct"]
    )
    return df
        
history_data = special_features(history_df)
current_data = special_features(curr_df)    
current_data.head()   

,season_id,game_id,game_date,team_id,tri_code,team_name,win,wins_before,games_before,win_pct_before,sos_past,sos_future,pp_opp,pp_opp_pg,pp_toi,pp_gf,pp_gpg,pp_net_g,pp_net_gpg,pp_pct,pp_net_pct,sh_ga,sh_ga_pg,times_sh,times_sh_pg,pk_net_pct,pk_pct,pk_net_g,pk_net_gpg,pk_toi_pg,pp_ga,pp_ga_pg,sh_gf,sh_gf_pg,c_pp_opps,c_pp_net_g,net_pp_pct,ppng_rank,c_pp_ga,c_times_sh,net_pk_pct,pk_rank,game_savePct,game_GAA,rush_shots_for,rush_shots_against,rush_shot_differential,c_rush_shots_for,c_rush_shots_against,c_rush_diff,rush_for_per_game,rush_against_per_game,rush_diff_per_game,shot_attempts_for,shot_attempts_against,shot_attempts_differential,corsi_for,corsi_against,corsi_diff,c_shots_diff,games_played,shot_attempts_for_per_game,shot_attempts_against_per_game,shots_diff_per_game,SCF,SCSF,SCGF,SCA,HDCF,MDCF,LDCF,HDCA,MDCA,LDCA,HDCF_pct,HDCA_pct,MDCF_pct,MDCA_pct,LDCF_pct,LDCA_pct,c_SCF,c_SCSF,c_SCGF,c_SCA,SCF_pg,SCA_pg,c_HDCF,c_HDCA,c_MDCF,c_MDCA,c_LDCF,c_LDCA,shots_for,hd_sog,md_sog,ld_sog,hd_goals,md_goals,ld_goals,shot_pct,shots_against,hd_against,md_against,ld_against,sv_pct,pdo,c_goals_for,c_hd_sog,c_md_sog,c_ld_sog,c_hd_goals,c_md_goals,c_ld_goals,c_goals_against,c_hd_against,c_md_against,c_ld_against,c_shot_pct,c_sv_pct,c_pdo,c_shots_for,c_shots_against,c_corsi_for,c_corsi_against,c_corsi_diff,corsi_diff_per_game,games_remaining,wins,wins_pctg,losses,ot_losses,rotw,rotw_pctg,rw,so_losses,so_wins,points,points_pct,projected_points,in_playoff_spot,gf_pctg,goal_diff,goal_diff_pctg,streak,streak_count,last_10_gp,last_10_wins,last_10_losses,last_10_otl,last_10_points,last_10_gf,last_10_ga,last_10_goal_diff,last_10_rotw,last_10_rw,home_gp,home_wins,home_losses,home_otlosses,home_rotw,home_rw,home_points,home_gf,home_ga,home_goal_diff,road_gp,road_wins,road_losses,road_ot_losses,road_rotw,road_rw,road_points,road_gf,road_ga,road_goal_diff,division,division_rank,division_home_rank,division_road_rank,division_last_10_rank,conference,conference_rank,conference_home_rank,conference_road_rank,conference_last_10_rank,league_rank,league_home_rank,league_road_rank,league_last_10_rank,wildcard_rank,goals_for,goals_against,ot_wins,overtimes,ot_games_pct,pt_pct_x_sos,gaa_x_sv,gd_x_rushdiff,pk_load_x_pk_pct
0,20252026,2025020001,2025-10-07,16,CHI,Chicago Blackhawks,0,<NA>,0,NaN,NaN,0.497453,3,3.0,360.0,0,0.0,0,0.0,0.00000,0.000000,0,0.0,2,2.0,0.50,0.50,-1,-1.0,222.0,1,1.0,0,0.0,3.0,0.0,0.0,4,1.0,2.0,50.0,4,0.91891,3.04740,2,8,-6,2,8,-6,2.0,8.0,-6.0,50,49,1,33,34,-1,1,1,50.0,49.0,1.0,28,19,2,65,11,14,26,8,21,23,0.578947,0.421053,0.400000,0.600000,0.530612,0.469388,28,19,2,65,28.0,65.0,11,8,14,21,26,23,19,3,5,12,0,2,0,10.526316,37,10,11,16,91.891892,102.418208,2,3,5,12,0,2,0,3,10,11,16,0.105263,0.918919,102.418208,50,49,33,34,-1,-1.0,81,0,0.0,1,0,0,0.0,0,0,0,0,0.0,0,0,2.0,-1,-1.0,L,1.0,1,0,1,0,0,2,3,-1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,2,3,-1,Central,8,1,8,8,Western,15,3,16,15,30,7,32,30,9,2,3,0,0,0.0,0.000000,2.800286,6,1.0
1,20252026,2025020001,2025-10-07,13,FLA,Florida Panthers,1,<NA>,0,NaN,NaN,0.497126,2,2.0,222.0,1,1.0,1,1.0,0.50000,0.500000,0,0.0,3,3.0,1.00,1.00,0,0.0,360.0,0,0.0,0,0.0,2.0,1.0,50.0,1,0.0,3.0,0.0,1,0.89473,2.00000,12,6,6,12,6,6,12.0,6.0,6.0,69,69,0,53,52,1,0,1,69.0,69.0,0.0,37,37,3,65,17,24,28,20,17,31,0.459459,0.540541,0.585366,0.414634,0.474576,0.525424,37,37,3,65,37.0,65.0,17,20,24,17,28,31,37,10,11,16,2,0,1,8.108108,19,3,5,12,89.473684,97.581792,3,10,11,16,2,0,1,2,3,5,12,0.081081,0.894737,97.581792,69,69,53,52,1,1.0,81,1,1.0,0,0,1,1.0,1,0,0,2,1.0,164,0,3.0,1,1.0,W,1.0,1,1,0,0,2,3,2,1,1,1,1,1,0,0,1,1,2,3,2,1,0,0,0,0,0,0,0,0,0,0,Atlantic,1,1,4,1,Eastern,2,1,7,2,3,1,12,3,0,3,2,0,0,0.0,0.497126,1.789460,6,3.0
2,20252026,2025020002,2025-10-07,3,NYR,New York Rangers,0,<NA>,0,NaN,NaN,0.513884,2,2.0,240.0,0,0.0,0,0.0,0.00000,0.000000,0,0.0,1,1.0,1.00,1.00,0,0.0,120.0,0,0.0,0,0.0,2.0,0.0,0.0,4,0.0,1.0,0.0,1,0.96551,1.00642,7,9,-2,7,9,-2,7.0,9.0,-2.0,60,60,0,36,36,0,0,1,60.0,60.0,0.0,34,25,0,66,14,19,29,16,19,24,0.466667,0.53333

In [ ]:
# con.execute("DROP TABLE IF EXISTS history_data")

# con.execute("DROP TABLE IF EXISTS current_data")

# con.execute("""
# CREATE TABLE IF NOT EXISTS history_data AS
# SELECT * FROM history_df
# """)

# con.execute("""
# CREATE TABLE IF NOT EXISTS current_data AS
# SELECT * FROM curr_df
# """)


In [182]:

history_df = con.execute('select* from history_data').fetch_df()
history_df.head(3)


,season_id,game_id,game_date,team_id,tri_code,team_name,win,wins_before,games_before,win_pct_before,sos_past,sos_future,pp_opp,pp_opp_pg,pp_toi,pp_gf,pp_gpg,pp_net_g,pp_net_gpg,pp_pct,pp_net_pct,sh_ga,sh_ga_pg,times_sh,times_sh_pg,pk_net_pct,pk_pct,pk_net_g,pk_net_gpg,pk_toi_pg,pp_ga,pp_ga_pg,sh_gf,sh_gf_pg,c_pp_opps,c_pp_net_g,net_pp_pct,ppng_rank,c_pp_ga,c_times_sh,net_pk_pct,pk_rank,game_savePct,game_GAA,rush_shots_for,rush_shots_against,rush_shot_differential,c_rush_shots_for,c_rush_shots_against,c_rush_diff,rush_for_per_game,rush_against_per_game,rush_diff_per_game,shot_attempts_for,shot_attempts_against,shot_attempts_differential,corsi_for,corsi_against,corsi_diff,c_shots_diff,games_played,shot_attempts_for_per_game,shot_attempts_against_per_game,shots_diff_per_game,SCF,SCSF,SCGF,SCA,HDCF,MDCF,LDCF,HDCA,MDCA,LDCA,HDCF_pct,HDCA_pct,MDCF_pct,MDCA_pct,LDCF_pct,LDCA_pct,c_SCF,c_SCSF,c_SCGF,c_SCA,SCF_pg,SCA_pg,c_HDCF,c_HDCA,c_MDCF,c_MDCA,c_LDCF,c_LDCA,shots_for,hd_sog,md_sog,ld_sog,hd_goals,md_goals,ld_goals,shot_pct,shots_against,hd_against,md_against,ld_against,sv_pct,pdo,c_goals_for,c_hd_sog,c_md_sog,c_ld_sog,c_hd_goals,c_md_goals,c_ld_goals,c_goals_against,c_hd_against,c_md_against,c_ld_against,c_shot_pct,c_sv_pct,c_pdo,c_shots_for,c_shots_against,c_corsi_for,c_corsi_against,c_corsi_diff,corsi_diff_per_game,games_remaining,wins,wins_pctg,losses,ot_losses,rotw,rotw_pctg,rw,so_losses,so_wins,points,points_pct,projected_points,in_playoff_spot,gf_pctg,goal_diff,goal_diff_pctg,streak,streak_count,last_10_gp,last_10_wins,last_10_losses,last_10_otl,last_10_points,last_10_gf,last_10_ga,last_10_goal_diff,last_10_rotw,last_10_rw,home_gp,home_wins,home_losses,home_otlosses,home_rotw,home_rw,home_points,home_gf,home_ga,home_goal_diff,road_gp,road_wins,road_losses,road_ot_losses,road_rotw,road_rw,road_points,road_gf,road_ga,road_goal_diff,division,division_rank,division_home_rank,division_road_rank,division_last_10_rank,conference,conference_rank,conference_home_rank,conference_road_rank,conference_last_10_rank,league_rank,league_home_rank,league_road_rank,league_last_10_rank,wildcard_rank,made_playoffs,goals_for,goals_against,ot_wins,overtimes,ot_games_pct,pt_pct_x_sos,gaa_x_sv,gd_x_rushdiff,pk_load_x_pk_pct
0,20132014,2013020001,2013-10-01,10,TOR,Toronto Maple Leafs,1,<NA>,0,NaN,NaN,0.496280,7,7.0,700.0,1,1.0,1,1.0,0.14285,0.142857,0,0.0,4,4.0,1.250000,1.00000,1,1.0,405.0,0,0.0,1,1.0,7.0,1.0,14.3,4,0.0,4.0,0.0,1,0.91891,3.01003,9,7,2,9,7,2,9.0,7.0,2.0,61,61,0,45.0,44.0,1.0,0,1,61.0,61.0,0.0,31,38,4,65.0,7,27,28,6,25,32,0.538462,0.461538,0.519231,0.480769,0.466667,0.533333,31,38,4,65.0,31.0,65.0,7,6,27,25,28,32,38,5,15,18,1,2,1,10.526316,37,11,9,18,91.891892,102.418208,4,5,15,18,1,2,1,3,11,9,18,0.105263,0.918919,102.418208,61,61,45.0,44.0,1.0,1.0,81,1,1.0,0,0,1,1.0,1,0,0,2,1.0,164,0,4.0,1,1.0,W,1.0,1,1,0,0,2,4,3,1,1,1,0,0,0,0,0,0,0,0,0,0,1,1,0,0,1,1,2,4,3,1,Atlantic,1,7,1,1,Eastern,1,14,1,1,2,25,1,2,0,0,4,3,0,0,0.0,0.49628,2.765947,2,4.00000
1,20132014,2013020001,2013-10-01,8,MTL,Montréal Canadiens,0,<NA>,0,NaN,NaN,0.486969,4,4.0,405.0,0,0.0,-1,-1.0,0.00000,-0.250000,1,1.0,7,7.0,0.857142,0.85714,-1,-1.0,700.0,1,1.0,0,0.0,4.0,-1.0,-25.0,6,1.0,7.0,14.3,2,0.89473,4.06549,7,9,-2,7,9,-2,7.0,9.0,-2.0,70,69,1,53.0,53.0,0.0,1,1,70.0,69.0,1.0,34,37,3,65.0,15,20,38,16,22,34,0.483871,0.516129,0.476190,0.523810,0.527778,0.472222,34,37,3,65.0,34.0,65.0,15,16,20,22,38,34,37,11,9,18,3,0,0,8.108108,38,5,15,18,89.473684,97.581792,3,11,9,18,3,0,0,4,5,15,18,0.081081,0.894737,97.581792,70,69,53.0,53.0,0.0,0.0,81,0,0.0,1,0,0,0.0,0,0,0,0,0.0,0,0,3.0,-1,-1.0,L,1.0,1,0,1,0,0,3,4,-1,0,0,1,0,1,0,0,0,0,3,4,-1,0,0,0,0,0,0,0,0,0,0,Atlantic,8,8,6,8,Eastern,15,16,8,15,29,30,17,29,9,1,3,4,0,0,0.0,0.00000,3.637516,2,5.99998
2,20132014,2013020002,2013-10-01,15,WSH,Washington Capitals,0,<NA>,0,NaN,NaN,0.480114,6,6.0,388.0,3,3.0,3,3.0,0.50000,0.500000,0,0.0,4,4.0,0.750000,0.75000,-1,-1.0,387.0,1,1.0,0,0.0,6.0,3.0,50.0,1,1.0,4.0,25.0,3,0.85294,5.06329,4,7,-3,4,7,-3,4.0,